<a href="https://colab.research.google.com/github/IgorKovacevicENNOH/ENNOH_Modelling/blob/main/Writing_in_PyPSA_results__dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction - Scenario 2026
packages installation

In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)


Mounted at /content/drive


In [2]:
!pip install openpyxl


In [3]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl

ROOT_DIR = os.getcwd()
PROJECT_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model")

In [4]:
sys.path.append(PROJECT_DIR)

from modules.getting_input_data import get_input_data

input_file_name = "input_file.xlsx"
input_data = get_input_data(PROJECT_DIR, input_file_name)

File found at: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/input_file.xlsx
The input data has been imported.


In [5]:
import json
import os

project_name = 'Europe'
zones = json.loads(input_data["zones"])
year = input_data["year"]
VALIDATION_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]), 'validation_dataset')
weather_profile = input_data["weather_profile"]

# Improved helper to strictly convert truthy/falsy values to 'TRUE'/'FALSE'
def to_bool_str(val):
    # If it's already a string like '1' or '0'
    if isinstance(val, (str, int, float)):
        s_val = str(val).lower()
        if s_val in ['1', '1.0', 'true']:
            return 'TRUE'
        if s_val in ['0', '0.0', 'false']:
            return 'FALSE'
    return 'TRUE' if bool(val) else 'FALSE'

prefix_results = f"FLEX_{to_bool_str(input_data['FLEX'])}_NTC_{to_bool_str(input_data['NTC'])}_H2_{to_bool_str(input_data['H2'])}_CH4_{to_bool_str(input_data['CH4'])}_Syn_{to_bool_str(input_data['Synthetic_fuels'])}"

# Constructing RESULT_DIR
RESULT_DIR = os.path.join(PROJECT_DIR, str(input_data["data_set"]),
    input_data['result_dir'], input_data["project_name"],
    input_data["scenario"], str(input_data["year"]), f'profile_{weather_profile}')


## Reading NT+

In [6]:
import os
import pandas as pd
import numpy as np

kpi_dashboard_path = os.path.join(VALIDATION_DIR, 'NT+_KPI_Dashboard.xlsx')

# Get all sheet names
excel_file = pd.ExcelFile(kpi_dashboard_path)
sheet_names = excel_file.sheet_names

all_sheets_data = {}
validation_dataset = {}

for sheet_name in sheet_names:
    # 1. Read the sheet
    df_raw = pd.read_excel(kpi_dashboard_path, sheet_name=sheet_name, header=None)

    # Skip sheets that are too small to contain our expected structure
    if len(df_raw) < 5:
        continue

    # 2. Extract column headers (Rows 3 and 4)
    headers_1 = df_raw.iloc[2].ffill().fillna('')
    headers_2 = df_raw.iloc[3].fillna('')

    cols = []
    for c1, c2 in zip(headers_1, headers_2):
        c1, c2 = str(c1).strip(), str(c2).strip()
        if c1.lower() == 'nan': c1 = ''
        if c2.lower() == 'nan': c2 = ''

        if c1 and c2: cols.append(f"{c1} - {c2}")
        elif c2: cols.append(c2)
        elif c1: cols.append(c1 if not c1.isnumeric() else f"Unnamed ({c1})")
        else: cols.append("Unnamed")

    # 3. Clean and format the dataframe
    df_clean = df_raw.iloc[4:].copy()

    # Ensure column count matches before assignment
    if len(cols) != len(df_clean.columns):
        print(f"Skipping sheet '{sheet_name}' due to column mismatch.")
        continue

    df_clean.columns = cols

    keys_a = df_clean.iloc[:, 0].replace('', np.nan).ffill().fillna('')
    keys_b = df_clean.iloc[:, 1].fillna('')
    combined_keys = [ka if ka == kb else f"{ka} - {kb}" if ka and kb else ka or kb or "Empty_Key" for ka, kb in zip(keys_a.astype(str).str.strip(), keys_b.astype(str).str.strip())]

    df_final = df_clean.iloc[:, 2:].copy()
    df_final.drop(columns=[c for c in df_final.columns if 'Unnamed' in str(c)], inplace=True)
    df_final['Excel_Row'] = df_clean.index + 1

    # Set Item_Key as the index to remove integer row numbers on the left
    df_final.index = combined_keys
    df_final.index.name = 'Item_Key'

    # 4. Define Data Dictionary (Major Categories)
    major_rows = [5, 48, 95, 138, 159, 166, 169]
    major_names = []
    valid_major_rows = []
    for r in major_rows:
        idx = r - 1
        if idx < len(df_raw):
            val_a, val_b = df_raw.iloc[idx, 0], df_raw.iloc[idx, 1]
            part_a = str(val_a).strip() if pd.notna(val_a) and str(val_a).strip().lower() != 'nan' else ""
            part_b = str(val_b).strip() if pd.notna(val_b) and str(val_b).strip().lower() != 'nan' else ""

            name = f"{part_a} - {part_b}" if part_a and part_b else part_a or part_b or f"Category_Row_{r}"
            major_names.append(name)
            valid_major_rows.append(r)

    data_dict = {}
    for i in range(len(valid_major_rows)):
        start_row = valid_major_rows[i] + 1
        end_row = valid_major_rows[i+1] - 1 if i + 1 < len(valid_major_rows) else df_final['Excel_Row'].max()
        data_dict[major_names[i]] = df_final[(df_final['Excel_Row'] >= start_row) & (df_final['Excel_Row'] <= end_row)].copy()

    # 5. Define Nested Dictionary (Sub-Categories)
    df_final['Sub_Category'] = df_raw.iloc[4:, 0].replace(r'^\s*$', np.nan, regex=True).ffill().fillna('').values
    nested_data_dict = {}

    for i in range(len(valid_major_rows)):
        start_row = valid_major_rows[i] + 1
        end_row = valid_major_rows[i+1] - 1 if i + 1 < len(valid_major_rows) else df_final['Excel_Row'].max()
        major_key = major_names[i]
        nested_data_dict[major_key] = {}

        df_section = df_final[(df_final['Excel_Row'] >= start_row) & (df_final['Excel_Row'] <= end_row)]
        for sub_cat in df_section['Sub_Category'].unique():
            sub_key = str(sub_cat).strip() or "General"
            nested_data_dict[major_key][sub_key] = df_section[df_section['Sub_Category'] == sub_cat].drop(columns=['Sub_Category']).copy()

    df_final = df_final.drop(columns=['Sub_Category'])

    all_sheets_data[sheet_name] = data_dict
    validation_dataset[sheet_name] = nested_data_dict

print(f"Successfully processed {len(validation_dataset)} sheets out of {len(sheet_names)} total sheets!")

Successfully processed 41 sheets out of 41 total sheets!


# Reading emply dataset

In [7]:
import os
import pandas as pd
import numpy as np

empty_dataset_path = os.path.join(VALIDATION_DIR, 'Empty_dataset.xlsx')

# Get all sheet names
excel_file = pd.ExcelFile(empty_dataset_path)
sheet_names = excel_file.sheet_names

all_sheets_data = {}
pypsa_dataset = {}

for sheet_name in sheet_names:
    # 1. Read the sheet
    df_raw = pd.read_excel(empty_dataset_path, sheet_name=sheet_name, header=None)

    # Skip sheets that are too small to contain our expected structure
    if len(df_raw) < 5:
        continue

    # 2. Extract column headers (Rows 3 and 4)
    headers_1 = df_raw.iloc[2].ffill().fillna('')
    headers_2 = df_raw.iloc[3].fillna('')

    cols = []
    for c1, c2 in zip(headers_1, headers_2):
        c1, c2 = str(c1).strip(), str(c2).strip()
        if c1.lower() == 'nan': c1 = ''
        if c2.lower() == 'nan': c2 = ''

        if c1 and c2: cols.append(f"{c1} - {c2}")
        elif c2: cols.append(c2)
        elif c1: cols.append(c1 if not c1.isnumeric() else f"Unnamed ({c1})")
        else: cols.append("Unnamed")

    # 3. Clean and format the dataframe
    df_clean = df_raw.iloc[4:].copy()

    # Ensure column count matches before assignment
    if len(cols) != len(df_clean.columns):
        print(f"Skipping sheet '{sheet_name}' due to column mismatch.")
        continue

    df_clean.columns = cols

    keys_a = df_clean.iloc[:, 0].replace('', np.nan).ffill().fillna('')
    keys_b = df_clean.iloc[:, 1].fillna('')
    combined_keys = [ka if ka == kb else f"{ka} - {kb}" if ka and kb else ka or kb or "Empty_Key" for ka, kb in zip(keys_a.astype(str).str.strip(), keys_b.astype(str).str.strip())]

    df_final = df_clean.iloc[:, 2:].copy()
    df_final.drop(columns=[c for c in df_final.columns if 'Unnamed' in str(c)], inplace=True)
    df_final['Excel_Row'] = df_clean.index + 1

    # Set Item_Key as the index to remove integer row numbers on the left
    df_final.index = combined_keys
    df_final.index.name = 'Item_Key'

    # 4. Define Data Dictionary (Major Categories)
    major_rows = [5, 48, 95, 138, 159, 166, 169]
    major_names = []
    valid_major_rows = []
    for r in major_rows:
        idx = r - 1
        if idx < len(df_raw):
            val_a, val_b = df_raw.iloc[idx, 0], df_raw.iloc[idx, 1]
            part_a = str(val_a).strip() if pd.notna(val_a) and str(val_a).strip().lower() != 'nan' else ""
            part_b = str(val_b).strip() if pd.notna(val_b) and str(val_b).strip().lower() != 'nan' else ""

            name = f"{part_a} - {part_b}" if part_a and part_b else part_a or part_b or f"Category_Row_{r}"
            major_names.append(name)
            valid_major_rows.append(r)

    data_dict = {}
    for i in range(len(valid_major_rows)):
        start_row = valid_major_rows[i] + 1
        end_row = valid_major_rows[i+1] - 1 if i + 1 < len(valid_major_rows) else df_final['Excel_Row'].max()
        data_dict[major_names[i]] = df_final[(df_final['Excel_Row'] >= start_row) & (df_final['Excel_Row'] <= end_row)].copy()

    # 5. Define Nested Dictionary (Sub-Categories)
    df_final['Sub_Category'] = df_raw.iloc[4:, 0].replace(r'^\s*$', np.nan, regex=True).ffill().fillna('').values
    nested_data_dict = {}

    for i in range(len(valid_major_rows)):
        start_row = valid_major_rows[i] + 1
        end_row = valid_major_rows[i+1] - 1 if i + 1 < len(valid_major_rows) else df_final['Excel_Row'].max()
        major_key = major_names[i]
        nested_data_dict[major_key] = {}

        df_section = df_final[(df_final['Excel_Row'] >= start_row) & (df_final['Excel_Row'] <= end_row)]
        for sub_cat in df_section['Sub_Category'].unique():
            sub_key = str(sub_cat).strip() or "General"
            nested_data_dict[major_key][sub_key] = df_section[df_section['Sub_Category'] == sub_cat].drop(columns=['Sub_Category']).copy()

    df_final = df_final.drop(columns=['Sub_Category'])

    all_sheets_data[sheet_name] = data_dict
    pypsa_dataset[sheet_name] = nested_data_dict

print(f"Successfully processed {len(pypsa_dataset)} sheets out of {len(sheet_names)} total sheets!")


Successfully processed 41 sheets out of 41 total sheets!


# Loading PyPSA Resutls

In [8]:
! pip install pypsa

In [9]:
network_path = os.path.join(RESULT_DIR, prefix_results)

In [10]:
import pypsa
import os

# Load the network from the corrected directory path
if os.path.exists(network_path):
    n_py = pypsa.Network(network_path)
    print(f"✅ Network successfully loaded from: {network_path}")
else:
    print(f"❌ Error: The directory {network_path} was not found.")

/usr/local/lib/python3.13/dist-packages/pypsa/network/io.py:2082: FutureWarning:

pandas infers the `str` dtype for string data since its version 3.0. PyPSA still converts it back to numpy object dtype on import, but will keep it from PyPSA 2.0 on. Set `pypsa.options.api.legacy_string_dtype` explicitly to suppress this warning.



✅ Network successfully loaded from: /content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/results/Europe/GA/2035/profile_3/FLEX_TRUE_NTC_TRUE_H2_TRUE_CH4_FALSE_Syn_FALSE


# Writing EU27 data to Pypsa Results

In [11]:
import pandas as pd
import numpy as np
import re

# 1. Load Validation Data from country-specific sheets
file_path = '/content/drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/TYNDP_scenario_2026/validation_dataset/NT+_KPI_Dashboard.xlsx'
xl = pd.ExcelFile(file_path)

# Identify country sheets (2-letter ISO codes)
country_sheets = [s for s in xl.sheet_names if len(s) == 2 and s.isupper()]

validation_records = []
for sheet in country_sheets:
    try:
        temp_df = pd.read_excel(xl, sheet_name=sheet, header=None)
        found_coords = np.where(temp_df.isin(['WS037']))

        if len(found_coords[0]) > 0:
            col_idx = found_coords[1][0]
            mask = temp_df.iloc[:, :5].apply(lambda row: row.astype(str).str.contains('hydrogen|h2', case=False, na=False).any(), axis=1)
            h2_row_indices = temp_df.index[mask].tolist()

            for row_idx in h2_row_indices:
                row_text = " ".join(temp_df.iloc[row_idx, :5].astype(str).tolist()).lower()
                if 'demand' in row_text or 'consumption' in row_text:
                    val = temp_df.iloc[row_idx, col_idx]
                    try:
                        val = float(val)
                        if not np.isnan(val):
                            validation_records.append({'Country': sheet, 'Validation Demand (TWh)': val})
                            break
                    except (ValueError, TypeError):
                        continue
    except Exception as e:
        continue

df_val_national = pd.DataFrame(validation_records)

# 2. Extract PyPSA H2 Demand directly from n_py
pypsa_records = []
if hasattr(n_py, 'loads') and not n_py.loads.empty:
    h2_loads = n_py.loads[n_py.loads.carrier.str.contains('H2|hydrogen', case=False, na=False)].copy()

    # Check if time-varying data exists
    has_time_varying = hasattr(n_py, 'loads_t') and 'p_set' in n_py.loads_t and not n_py.loads_t.p_set.empty

    for load_name, row in h2_loads.iterrows():
        country = str(row['bus'])[:2]
        demand_mwh = 0

        # Use snapshot weightings if available, else assume 1 hour per snapshot
        weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') else 1

        # First try fetching from time series data
        if has_time_varying and load_name in n_py.loads_t.p_set.columns:
            time_series = n_py.loads_t.p_set[load_name]
            if time_series.sum() > 0:
                demand_mwh = (time_series * weightings).sum()

        # Fallback to static if time series is 0 or missing
        if demand_mwh == 0:
            hours = len(n_py.snapshots) if hasattr(n_py, 'snapshots') else 8760
            demand_mwh = row['p_set'] * hours

        pypsa_records.append({'Country': country, 'PyPSA Demand (TWh)': demand_mwh / 1e6})

pypsa_national = pd.DataFrame(pypsa_records)
if not pypsa_national.empty:
    # Group by country in case of multiple H2 load buses per country
    pypsa_national = pypsa_national.groupby('Country', as_index=False)['PyPSA Demand (TWh)'].sum()
else:
    pypsa_national = pd.DataFrame(columns=['Country', 'PyPSA Demand (TWh)'])

print(f"Extracted {len(pypsa_national)} PyPSA records and {len(df_val_national)} Validation records.")
if len(pypsa_national) > 0:
    print("PyPSA Countries:", pypsa_national['Country'].unique())
if len(df_val_national) > 0:
    print("Validation Countries:", df_val_national['Country'].unique())

Extracted 27 PyPSA records and 0 Validation records.
PyPSA Countries: <ArrowStringArray>
['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR',
 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'NL', 'PL', 'PT', 'RO', 'SE', 'SI', 'SK',
 'UK']
Length: 27, dtype: str


In [12]:
# 3. Merge and Compare
if not pypsa_national.empty and not df_val_national.empty:
    comparison_df = pypsa_national.merge(
        df_val_national,
        on='Country',
        how='inner'
    )

    # Round initial demand columns to 2 decimal places
    comparison_df['PyPSA Demand (TWh)'] = comparison_df['PyPSA Demand (TWh)'].round(2)
    comparison_df['Validation Demand (TWh)'] = comparison_df['Validation Demand (TWh)'].round(2)

    # Calculate deltas and percentages
    comparison_df['Delta (TWh)'] = (comparison_df['PyPSA Demand (TWh)'] - comparison_df['Validation Demand (TWh)']).round(2)
    comparison_df['Diff (%)'] = (comparison_df['Delta (TWh)'] / comparison_df['Validation Demand (TWh)'] * 100).round(2)

    print(f"=== National H2 Demand Comparison (Year 2035, Profile 2) ===")
    display(comparison_df.sort_values(by='PyPSA Demand (TWh)', ascending=False))
else:
    print("Error: One of the dataframes is empty. Check parsing logic.")
    print(f"PyPSA National Rows: {len(pypsa_national)}")
    print(f"Validation National Rows: {len(df_val_national)}")

Error: One of the dataframes is empty. Check parsing logic.
PyPSA National Rows: 27
Validation National Rows: 0


In [13]:
import pandas as pd
import numpy as np

print("Extracting Hydrogen Installed Capacity [GW]...")

# 1. Validation Data Extraction (Excel Rows 29 to 47 -> Pandas index 28 to 46)
val_cap_records = []
for sheet in country_sheets:
    try:
        temp_df = pd.read_excel(xl, sheet_name=sheet, header=None)
        found_coords = np.where(temp_df.isin(['WS037']))

        if len(found_coords[0]) > 0:
            col_idx = found_coords[1][0]
            h2_cap_sum = 0.0
            # Rows 29 to 47 in Excel correspond to indices 28 to 46 in pandas
            for row_idx in range(28, 47):
                if row_idx < len(temp_df):
                    val = temp_df.iloc[row_idx, col_idx]
                    try:
                        val = float(val)
                        if not np.isnan(val):
                            h2_cap_sum += val
                    except (ValueError, TypeError):
                        continue
            val_cap_records.append({'Country': sheet, 'Validation H2 Cap (GW)': h2_cap_sum})
    except Exception as e:
        continue

df_val_cap = pd.DataFrame(val_cap_records)

# 2. PyPSA Data Extraction
pypsa_cap_records = []
if hasattr(n_py, 'links') and not n_py.links.empty:
    # Filter for H2 components (Electrolyzers, fuel cells, SMR, etc.)
    h2_links = n_py.links[n_py.links.carrier.str.contains('H2|hydrogen|electrolyser|fuel cell', case=False, na=False)].copy()

    # Exclude pipelines, retrofits, and storage handling links to isolate generation/conversion capacity
    exclude_patterns = 'pipe|retro|store|storage|compressor'
    h2_links = h2_links[~h2_links.index.str.contains(exclude_patterns, case=False, na=False)]

    def is_domestic(row):
        b0 = str(row['bus0'])[:2]
        b1 = str(row['bus1'])[:2] if pd.notna(row['bus1']) and str(row['bus1']).strip() != '' else b0
        return b0 == b1

    h2_links['Domestic'] = h2_links.apply(is_domestic, axis=1)
    domestic_h2_links = h2_links[h2_links['Domestic'] == True]

    for idx, row in domestic_h2_links.iterrows():
        country = str(row['bus0'])[:2]
        cap_gw = row.get('p_nom_opt', row.get('p_nom', 0)) / 1000.0  # Convert MW to GW
        pypsa_cap_records.append({'Country': country, 'PyPSA H2 Cap (GW)': cap_gw})

df_pypsa_cap = pd.DataFrame(pypsa_cap_records)
if not df_pypsa_cap.empty:
    df_pypsa_cap = df_pypsa_cap.groupby('Country', as_index=False)['PyPSA H2 Cap (GW)'].sum()
else:
    df_pypsa_cap = pd.DataFrame(columns=['Country', 'PyPSA H2 Cap (GW)'])

# 3. Merge and Compare
if not df_pypsa_cap.empty or not df_val_cap.empty:
    comp_cap_df = df_pypsa_cap.merge(df_val_cap, on='Country', how='outer').fillna(0)
    comp_cap_df['PyPSA H2 Cap (GW)'] = comp_cap_df['PyPSA H2 Cap (GW)'].round(3)
    comp_cap_df['Validation H2 Cap (GW)'] = comp_cap_df['Validation H2 Cap (GW)'].round(3)
    comp_cap_df['Delta (GW)'] = (comp_cap_df['PyPSA H2 Cap (GW)'] - comp_cap_df['Validation H2 Cap (GW)']).round(3)

    print("\n=== National H2 Installed Capacity [GW] Comparison (Rows 29-47) ===")
    display(comp_cap_df.sort_values(by='PyPSA H2 Cap (GW)', ascending=False).reset_index(drop=True))
else:
    print("No H2 capacity data found to compare.")


Extracting Hydrogen Installed Capacity [GW]...

=== National H2 Installed Capacity [GW] Comparison (Rows 29-47) ===


,Country,PyPSA H2 Cap (GW),Validation H2 Cap (GW),Delta (GW)
0,DE,42.889,247.190,-204.301
1,SE,15.502,66.036,-50.534
2,UK,13.205,39.157,-25.952
3,DK,10.946,19.539,-8.593
4,NL,9.072,67.199,-58.127
5,FR,8.000,56.230,-48.230
6,ES,7.300,57.066,-49.766
7,FI,5.500,75.658,-70.158
8,IT,5.175,52.199,-47.024
9,PL,3.225,34.409,-31.184


In [14]:
import pandas as pd
import numpy as np

print("Extracting Detailed Hydrogen Technologies Capacity [GW]...")

# 1. Validation Detail Extraction (Rows 28 to 46)
val_tech_records = []
for sheet in country_sheets:
    try:
        temp_df = pd.read_excel(xl, sheet_name=sheet, header=None)
        found_coords = np.where(temp_df.isin(['WS037']))

        if len(found_coords[0]) > 0:
            col_idx = found_coords[1][0]
            for row_idx in range(28, 47):
                if row_idx < len(temp_df):
                    label_part1 = str(temp_df.iloc[row_idx, 0]).strip()
                    label_part2 = str(temp_df.iloc[row_idx, 1]).strip()

                    if label_part1.lower() == 'nan': label_part1 = ""
                    if label_part2.lower() == 'nan': label_part2 = ""

                    label = f"{label_part1} - {label_part2}" if label_part1 else label_part2
                    if not label: continue

                    val = temp_df.iloc[row_idx, col_idx]
                    try:
                        val = float(val)
                        if not np.isnan(val) and val != 0:
                            val_tech_records.append({'Country': sheet, 'Technology': label, 'Validation Cap (GW)': val})
                    except (ValueError, TypeError):
                        continue
    except Exception as e:
        continue

df_val_tech = pd.DataFrame(val_tech_records)

# 2. PyPSA Detail Extraction & Technology Mapping
pypsa_tech_records = []

# Mapping function to assign PyPSA components to Validation Dashboard Categories
def map_pypsa_tech(name):
    n = str(name).lower()
    if 'electrolyser' in n:
        # Check if connected to dedicated renewables (DRES/SRES)
        if 'dres' in n or 'sres' in n or 'dedicated' in n:
            return 'H2 - Electrolyser Dedicated'
        return 'H2 - Electrolyser Grid'
    elif 'smr' in n:
        if 'blue' in n or 'ccs' in n:
            return 'H2 - SMR (Blue) and Pyrolisis'
        return 'H2 - SMR (Grey)'
    elif 'fuel cell' in n or 'fc' in n:
        return 'H2 - Fuel Cell'
    else:
        return 'H2 - Other'

if hasattr(n_py, 'links') and not n_py.links.empty:
    h2_links_detailed = n_py.links[n_py.links.carrier.str.contains('H2|hydrogen|electrolyser|fuel cell', case=False, na=False)].copy()

    def is_domestic(row):
        b0 = str(row['bus0'])[:2]
        b1 = str(row['bus1'])[:2] if pd.notna(row['bus1']) and str(row['bus1']).strip() != '' else b0
        return b0 == b1

    h2_links_detailed['Domestic'] = h2_links_detailed.apply(is_domestic, axis=1)
    h2_links_dom = h2_links_detailed[h2_links_detailed['Domestic'] == True].copy()

    h2_links_dom['Technology'] = h2_links_dom.index.map(map_pypsa_tech)
    h2_links_dom['Cap_GW'] = h2_links_dom.apply(lambda r: r.get('p_nom_opt', r.get('p_nom', 0)) / 1000.0, axis=1)

    for idx, row in h2_links_dom.iterrows():
        country = str(row['bus0'])[:2]
        pypsa_tech_records.append({'Country': country, 'Technology': row['Technology'], 'PyPSA Cap (GW)': row['Cap_GW']})

df_pypsa_tech = pd.DataFrame(pypsa_tech_records)
if not df_pypsa_tech.empty:
    df_pypsa_tech = df_pypsa_tech.groupby(['Country', 'Technology'], as_index=False)['PyPSA Cap (GW)'].sum()
else:
    df_pypsa_tech = pd.DataFrame(columns=['Country', 'Technology', 'PyPSA Cap (GW)'])

# 3. Merge PyPSA and Validation Data
df_tech_comp = pd.merge(df_val_tech, df_pypsa_tech, on=['Country', 'Technology'], how='outer').fillna(0)

# 4. EU27 Aggregation
# Standard EU27 list
eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']
df_tech_comp_eu27 = df_tech_comp[df_tech_comp['Country'].isin(eu27_countries)].groupby('Technology', as_index=False)[['Validation Cap (GW)', 'PyPSA Cap (GW)']].sum()
df_tech_comp_eu27['Country'] = 'EU27'

# 5. Combine Country Level and EU27
df_final_tech = pd.concat([df_tech_comp_eu27, df_tech_comp], ignore_index=True)

# Calculate Deltas
df_final_tech['Delta (GW)'] = (df_final_tech['PyPSA Cap (GW)'] - df_final_tech['Validation Cap (GW)']).round(3)
df_final_tech['PyPSA Cap (GW)'] = df_final_tech['PyPSA Cap (GW)'].round(3)
df_final_tech['Validation Cap (GW)'] = df_final_tech['Validation Cap (GW)'].round(3)

# Filter out rows where both are 0 to keep the table clean
df_final_tech = df_final_tech[(df_final_tech['PyPSA Cap (GW)'] != 0) | (df_final_tech['Validation Cap (GW)'] != 0)]

print("\n=== EU27 Aggregated Breakdown ===")
display(df_final_tech[df_final_tech['Country'] == 'EU27'].sort_values('PyPSA Cap (GW)', ascending=False).set_index('Technology').drop(columns=['Country']))

print("\n=== Country-level Breakdown ===")
display(df_final_tech[df_final_tech['Country'] != 'EU27'].sort_values(['Country', 'Technology']).set_index(['Country', 'Technology']))


Extracting Detailed Hydrogen Technologies Capacity [GW]...

=== EU27 Aggregated Breakdown ===


,Validation Cap (GW),PyPSA Cap (GW),Delta (GW)
Technology,,,
H2 - Other,0.000,2742.564,2742.564
H2 - Electrolyser Grid,0.000,97.726,97.726
EU Cross-Border Pipeline Capacity (Import),273.958,0.000,-273.958
EU Cross-Border Pipeline Capacity (Export),273.958,0.000,-273.958
E-Liquids Capacity (from H2),14.162,0.000,-14.162
Electrolyzers E-Market (P2G) [GW Elec],96.852,0.000,-96.852
Electrolyzers DRES (P2G) [GW Elec],12.825,0.000,-12.825
Electrolyzers SRES (P2G) [GW Elec],30.257,0.000,-30.257
H2 - SMR (Blue) and Pyrolisis,11.662,0.000,-11.662



=== Country-level Breakdown ===


Validation Cap (GW)  \
Country Technology                                                        
AT      EU Cross-Border Pipeline Capacity (Export)               14.831   
        EU Cross-Border Pipeline Capacity (Import)               16.314   
        Electrolyzers E-Market (P2G) [GW Elec]                    3.050   
        H2 - Electrolyser Grid                                    0.000   
BA      EU Cross-Border Pipeline Capacity (Export)                4.061   
...                                                                 ...   
UK      H2 - Other                                                0.000   
        H2 - SMR (Blue) and Pyrolisis                             5.228   
        H2 Storage Injection Capacity                            10.987   
        H2 Storage WGV [TWh]                                      2.637   
        H2 Storage Withdrawal Capacity [GW]                      10.987   

                                                    PyPSA Cap (GW)  Delta (GW)  
Country Technology                                                              
AT      EU Cross-Border Pipeline Capacity (Export)           0.000     -14.831  
        EU Cross-Border Pipeline Capacity (Import)           0.000     -16.314  
        Electrolyzers E-Market (P2G) [GW Elec]               0.000      -3.050  
        H2 - Electrolyser Grid                               3.050       3.050  
BA      EU Cross-Border Pipeline Capacity (Export)           0.000      -4.061  
...                                                            ...         ...  
UK      H2 - Other                                           3.887       3.887  
        H2 - SMR (Blue) and Pyrolisis                        0.000      -5.228  
        H2 Storage Injection Capacity                        0.000     -10.987  
        H2 Storage WGV [TWh]                                 0.000      -2.637  
        H2 Storage Withdrawal Capacity [GW]                  0.000     -10.987  

[191 rows x 3 columns]

### SMR and Electrolysers

In [15]:
import pandas as pd
import numpy as np

print("Focusing on the 3 Electrolyzer Technologies...")

# 1. Validation Data Filter (Using the exact names from the dashboard)
target_techs = [
    'Electrolyzers E-Market (P2G) [GW Elec]',
    'Electrolyzers DRES (P2G) [GW Elec]',
    'Electrolyzers SRES (P2G) [GW Elec]'
]
# Make sure df_val_tech exists from previous cell
df_val_elec = df_val_tech[df_val_tech['Technology'].isin(target_techs)].copy()

# 2. PyPSA Data Extraction tailored for these 3 categories
pypsa_elec_records = []
if hasattr(n_py, 'links') and not n_py.links.empty:
    # Correctly filter by checking if the index name contains 'electrolyser'
    # (carrier is typically 'H2', so filtering by carrier fails)
    elec_links = n_py.links[n_py.links.index.str.contains('electrolyser|electrolyzer', case=False, na=False)].copy()

    for idx, row in elec_links.iterrows():
        n = str(idx).lower()
        # Map PyPSA component names to Validation Categories
        if 'dres' in n:
            tech = 'Electrolyzers DRES (P2G) [GW Elec]'
        elif 'sres' in n:
            tech = 'Electrolyzers SRES (P2G) [GW Elec]'
        else:
            tech = 'Electrolyzers E-Market (P2G) [GW Elec]'

        # Extract country: For DRES/SRES, bus0 is 'SRES_DEh2' etc., so bus0[:2] yields 'SR'.
        # However, bus1 is the H2 bus (e.g. 'DEh2'), which reliably starts with the country code.
        country = str(row['bus1'])[:2] if pd.notna(row['bus1']) and str(row['bus1']).strip() != '' else str(row['bus0'])[:2]

        # Assume electrolyzers are domestic, get capacity in GW
        cap = row.get('p_nom_opt', row.get('p_nom', 0)) / 1000.0
        pypsa_elec_records.append({'Country': country, 'Technology': tech, 'PyPSA Cap (GW)': cap})

df_pypsa_elec = pd.DataFrame(pypsa_elec_records)
if not df_pypsa_elec.empty:
    df_pypsa_elec = df_pypsa_elec.groupby(['Country', 'Technology'], as_index=False)['PyPSA Cap (GW)'].sum()
else:
    df_pypsa_elec = pd.DataFrame(columns=['Country', 'Technology', 'PyPSA Cap (GW)'])

# 3. Merge
df_elec_comp = pd.merge(df_val_elec, df_pypsa_elec, on=['Country', 'Technology'], how='outer').fillna(0)

# 4. EU27 Aggregation
eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']
df_elec_eu27 = df_elec_comp[df_elec_comp['Country'].isin(eu27_countries)].groupby('Technology', as_index=False)[['Validation Cap (GW)', 'PyPSA Cap (GW)']].sum()
df_elec_eu27['Country'] = 'EU27'

# 5. Combine and Format
df_final_elec = pd.concat([df_elec_eu27, df_elec_comp], ignore_index=True)
df_final_elec['Delta (GW)'] = (df_final_elec['PyPSA Cap (GW)'] - df_final_elec['Validation Cap (GW)']).round(3)
df_final_elec['PyPSA Cap (GW)'] = df_final_elec['PyPSA Cap (GW)'].round(3)
df_final_elec['Validation Cap (GW)'] = df_final_elec['Validation Cap (GW)'].round(3)

# Remove rows where everything is zero
df_final_elec = df_final_elec[(df_final_elec['PyPSA Cap (GW)'] != 0) | (df_final_elec['Validation Cap (GW)'] != 0)]

print("\n=== Electrolyzers: EU27 Aggregated ===")
display(df_final_elec[df_final_elec['Country'] == 'EU27'].set_index('Technology').drop(columns=['Country']))

print("\n=== Electrolyzers: Country Level (Pivot Table) ===")
# Pivot table for easier country-by-country reading
country_elec = df_final_elec[df_final_elec['Country'] != 'EU27'].copy()
if not country_elec.empty:
    pivot_elec = country_elec.pivot(index='Country', columns='Technology', values=['Validation Cap (GW)', 'PyPSA Cap (GW)', 'Delta (GW)'])
    # Flatten the MultiIndex columns for presentation
    pivot_elec.columns = [f"{tech} - {metric}" for metric, tech in pivot_elec.columns]
    pivot_elec = pivot_elec.fillna(0).sort_index()
    display(pivot_elec)
else:
    print("No country-level data to display.")


Focusing on the 3 Electrolyzer Technologies...

=== Electrolyzers: EU27 Aggregated ===


,Validation Cap (GW),PyPSA Cap (GW),Delta (GW)
Technology,,,
Electrolyzers DRES (P2G) [GW Elec],12.825,12.825,0.000
Electrolyzers E-Market (P2G) [GW Elec],96.852,97.726,0.874
Electrolyzers SRES (P2G) [GW Elec],30.257,30.257,0.000



=== Electrolyzers: Country Level (Pivot Table) ===


,Electrolyzers DRES (P2G) [GW Elec] - Validation Cap (GW),Electrolyzers E-Market (P2G) [GW Elec] - Validation Cap (GW),Electrolyzers SRES (P2G) [GW Elec] - Validation Cap (GW),Electrolyzers DRES (P2G) [GW Elec] - PyPSA Cap (GW),Electrolyzers E-Market (P2G) [GW Elec] - PyPSA Cap (GW),Electrolyzers SRES (P2G) [GW Elec] - PyPSA Cap (GW),Electrolyzers DRES (P2G) [GW Elec] - Delta (GW),Electrolyzers E-Market (P2G) [GW Elec] - Delta (GW),Electrolyzers SRES (P2G) [GW Elec] - Delta (GW)
Country,,,,,,,,,
AT,0.000,3.050,0.000,0.000,3.050,0.000,0.0,0.000,0.0
BE,0.000,0.150,0.000,0.000,0.150,0.000,0.0,0.000,0.0
BG,0.000,0.872,0.000,0.000,0.872,0.000,0.0,0.000,0.0
CH,0.000,0.988,0.000,0.000,0.988,0.000,0.0,0.000,0.0
CY,0.000,0.041,0.000,0.000,0.041,0.000,0.0,0.000,0.0
CZ,0.000,0.128,0.000,0.000,0.128,0.000,0.0,0.000,0.0
DE,1.000,25.936,0.921,1.000,25.936,0.921,0.0,0.000,-0.0
DK,0.000,10.072,0.000,0.000,10.946,0.000,0.0,0.874,0.0
EE,0.000,0.100,0.000,0.000,0.100,0.000,0.0,0.000,0.0


In [16]:
print("--- Debugging SMR / Blue / Grey Hydrogen Components ---")

# Keywords to search for in component names
keywords = 'smr|blue|grey|pyrol|reform|methane to h2|ch4 to h2'

# 1. Check Links
if hasattr(n_py, 'links') and not n_py.links.empty:
    suspect_links = n_py.links[n_py.links.index.str.contains(keywords, case=False, na=False)]
    print(f"\n1. Links matching keywords ({keywords}): {len(suspect_links)} found.")
    if not suspect_links.empty:
        display(suspect_links[['bus0', 'bus1', 'carrier', 'p_nom', 'p_nom_opt']].head(10))

    # Let's also just list all unique link carriers to see if there's a dedicated carrier for them
    print("\nUnique Link carriers in the network:")
    print(n_py.links.carrier.unique())

# 2. Check Generators (Sometimes SMR is modeled as a generator rather than a link)
if hasattr(n_py, 'generators') and not n_py.generators.empty:
    suspect_gens = n_py.generators[n_py.generators.index.str.contains(keywords, case=False, na=False)]
    print(f"\n2. Generators matching keywords: {len(suspect_gens)} found.")
    if not suspect_gens.empty:
        display(suspect_gens[['bus', 'carrier', 'p_nom', 'p_nom_opt']].head(10))

    print("\nUnique Generator carriers in the network:")
    print(n_py.generators.carrier.unique())

--- Debugging SMR / Blue / Grey Hydrogen Components ---

1. Links matching keywords (smr|blue|grey|pyrol|reform|methane to h2|ch4 to h2): 11 found.


,bus0,bus1,carrier,p_nom,p_nom_opt
name,,,,,
SMR_b_BE00_gas_bus_BEh2,BE00_gas_bus,BEh2,natural_gas,3076.923077,3076.923077
SMR_g_DE00_gas_busDEh2Z1,DE00_gas_bus,DEh2Z1,natural_gas,6294.285714,6294.285714
SMR_b_DE00_gas_bus_DEh2,DE00_gas_bus,DEh2,natural_gas,8163.076923,8163.076923
SMR_b_FI00_gas_bus_FIh2,FI00_gas_bus,FIh2,natural_gas,1039.353846,1039.353846
SMR_b_FR00_gas_bus_FRh2,FR00_gas_bus,FRh2,natural_gas,430.769231,430.769231
SMR_b_ITN1_gas_bus_ITh2,ITN1_gas_bus,ITh2,natural_gas,646.153846,646.153846
SMR_g_LT00_gas_busLTh2Z1,LT00_gas_bus,LTh2Z1,natural_gas,2285.714286,2285.714286
SMR_b_NL00_gas_bus_NLh2,NL00_gas_bus,NLh2,natural_gas,3692.307692,3692.307692
SMR_b_PL00_gas_bus_PLh2,PL00_gas_bus,PLh2,natural_gas,892.307692,892.307692



Unique Link carriers in the network:
['natural_gas' 'oil' 'lignite' 'hard_coal' 'AC' 'H2' 'cross_border']

2. Generators matching keywords: 0 found.

Unique Generator carriers in the network:
['hard_coal' 'lignite' 'oil' 'natural_gas' 'AC' 'other_non_res' 'nuclear'
 'solar' 'wind_onshore' 'hydro' 'RES' 'wind_offshore' 'solar_thermal' 'H2'
 'import' 'H2_import']


In [17]:
import pandas as pd

print("Focusing specifically on SMR (Blue and Grey) Capacities [GW]...")

# Filter the already combined dataframe for just SMR technologies
smr_techs = [
    'H2 - SMR (Blue) and Pyrolisis',
    'H2 - SMR (Grey)'
]

# df_final_tech contains the merged results from the previous cells
df_smr_only = df_final_tech[df_final_tech['Technology'].isin(smr_techs)].copy()

print("\n=== SMR (Blue & Grey): EU27 Aggregated ===")
display(df_smr_only[df_smr_only['Country'] == 'EU27'].set_index('Technology').drop(columns=['Country']))

print("\n=== SMR (Blue & Grey): Country Level ===")
country_smr = df_smr_only[df_smr_only['Country'] != 'EU27'].copy()
if not country_smr.empty:
    # Pivot table for a clean side-by-side comparison per country
    pivot_smr = country_smr.pivot(index='Country', columns='Technology', values=['Validation Cap (GW)', 'PyPSA Cap (GW)', 'Delta (GW)'])
    pivot_smr.columns = [f"{tech} - {metric}" for metric, tech in pivot_smr.columns]
    pivot_smr = pivot_smr.fillna(0).sort_index()
    display(pivot_smr)
else:
    print("No country-level SMR data to display.")

Focusing specifically on SMR (Blue and Grey) Capacities [GW]...

=== SMR (Blue & Grey): EU27 Aggregated ===


,Validation Cap (GW),PyPSA Cap (GW),Delta (GW)
Technology,,,
H2 - SMR (Blue) and Pyrolisis,11.662,0.0,-11.662



=== SMR (Blue & Grey): Country Level ===


,H2 - SMR (Blue) and Pyrolisis - Validation Cap (GW),H2 - SMR (Blue) and Pyrolisis - PyPSA Cap (GW),H2 - SMR (Blue) and Pyrolisis - Delta (GW)
Country,,,
BE,2.000,0.0,-2.000
DE,5.306,0.0,-5.306
FI,0.676,0.0,-0.676
FR,0.280,0.0,-0.280
IT,0.420,0.0,-0.420
NL,2.400,0.0,-2.400
PL,0.580,0.0,-0.580
UK,5.228,0.0,-5.228


In [18]:
import pandas as pd

print("=== Validation Dataset: SMR Capacities per Country (GW) ===")

# df_smr_only was created in cell 437b5323 and contains both Blue and Grey SMRs
smr_validation = df_smr_only[df_smr_only['Country'] != 'EU27'].copy()

# Sum both 'Blue' and 'Grey' SMR capacities for each country
smr_grouped = smr_validation.groupby('Country', as_index=False)['Validation Cap (GW)'].sum()

# Filter out countries with 0 GW to keep the list clean, and sort descending
smr_grouped = smr_grouped[smr_grouped['Validation Cap (GW)'] > 0].sort_values('Validation Cap (GW)', ascending=False).reset_index(drop=True)

display(smr_grouped)


=== Validation Dataset: SMR Capacities per Country (GW) ===


,Country,Validation Cap (GW)
0,DE,5.306
1,UK,5.228
2,NL,2.400
3,BE,2.000
4,FI,0.676
5,PL,0.580
6,IT,0.420
7,FR,0.280


In [19]:
print("Adding SMR and SMR_g links based on validation capacities...")

added_smr_links = []

for idx, row in smr_validation.iterrows():
    country = row['Country']
    tech = row['Technology']
    cap_gw = row['Validation Cap (GW)']

    if cap_gw <= 0:
        continue

    cap_mw = cap_gw * 1000.0

    # Determine link name based on technology
    if 'Blue' in tech:
        link_name = f"SMR_{country}"
        carrier_name = "SMR"
    else:
        link_name = f"SMR_g_{country}"
        carrier_name = "SMR_g"

    gas_bus = f"{country}00_gas_bus"
    h2_bus = f"{country}h2"

    # Ensure buses exist before adding the link
    if gas_bus not in n_py.buses.index:
        n_py.add("Bus", gas_bus, carrier="natural_gas", location=country)
    if h2_bus not in n_py.buses.index:
        n_py.add("Bus", h2_bus, carrier="H2", location=country)

    # Add or update the link in PyPSA
    if link_name in n_py.links.index:
        n_py.links.loc[link_name, 'p_nom'] += cap_mw
        n_py.links.loc[link_name, 'p_nom_opt'] += cap_mw
    else:
        n_py.add(
            "Link",
            link_name,
            bus0=gas_bus,
            bus1=h2_bus,
            carrier=carrier_name,
            p_nom=cap_mw,
            p_nom_opt=cap_mw,
            efficiency=0.75  # Approximate standard SMR efficiency
        )

    added_smr_links.append({
        'Link Name': link_name,
        'Country': country,
        'Technology': tech,
        'Capacity Added (MW)': cap_mw
    })

df_added_smr = pd.DataFrame(added_smr_links)
if not df_added_smr.empty:
    display(df_added_smr)
    print(f"\nSuccessfully added/updated {len(df_added_smr)} SMR links in the PyPSA network.")
else:
    print("No positive SMR capacities found to add.")

Adding SMR and SMR_g links based on validation capacities...


,Link Name,Country,Technology,Capacity Added (MW)
0,SMR_BE,BE,H2 - SMR (Blue) and Pyrolisis,2000.0
1,SMR_DE,DE,H2 - SMR (Blue) and Pyrolisis,5306.0
2,SMR_FI,FI,H2 - SMR (Blue) and Pyrolisis,676.0
3,SMR_FR,FR,H2 - SMR (Blue) and Pyrolisis,280.0
4,SMR_IT,IT,H2 - SMR (Blue) and Pyrolisis,420.0
5,SMR_NL,NL,H2 - SMR (Blue) and Pyrolisis,2400.0
6,SMR_PL,PL,H2 - SMR (Blue) and Pyrolisis,580.0
7,SMR_UK,UK,H2 - SMR (Blue) and Pyrolisis,5228.0



Successfully added/updated 8 SMR links in the PyPSA network.


In [20]:
import pandas as pd

print("Re-evaluating SMR capacities in PyPSA after updates...")

# Extract updated SMR capacities from PyPSA
updated_smr_records = []
if hasattr(n_py, 'links') and not n_py.links.empty:
    # Filter for SMR links (carrier = SMR or SMR_g)
    smr_links = n_py.links[n_py.links.carrier.isin(['SMR', 'SMR_g'])].copy()

    for idx, row in smr_links.iterrows():
        # Get country from bus0 (e.g. BE00_gas_bus -> BE)
        country = str(row['bus0'])[:2]

        # Map carrier back to Technology name
        if row['carrier'] == 'SMR':
            tech = 'H2 - SMR (Blue) and Pyrolisis'
        else:
            tech = 'H2 - SMR (Grey)'

        # Convert MW back to GW
        cap_gw = row.get('p_nom_opt', row.get('p_nom', 0)) / 1000.0
        updated_smr_records.append({'Country': country, 'Technology': tech, 'PyPSA Cap (GW)': cap_gw})

df_pypsa_smr_updated = pd.DataFrame(updated_smr_records)
if not df_pypsa_smr_updated.empty:
    df_pypsa_smr_updated = df_pypsa_smr_updated.groupby(['Country', 'Technology'], as_index=False)['PyPSA Cap (GW)'].sum()
else:
    df_pypsa_smr_updated = pd.DataFrame(columns=['Country', 'Technology', 'PyPSA Cap (GW)'])

# Get the original validation data for SMR (from df_smr_only)
df_val_smr_only = df_smr_only[['Country', 'Technology', 'Validation Cap (GW)']].copy()

# Merge
df_smr_updated_comp = pd.merge(df_val_smr_only, df_pypsa_smr_updated, on=['Country', 'Technology'], how='outer').fillna(0)

# Force both technologies to appear even if they are 0 for the layout
smr_techs = ['H2 - SMR (Blue) and Pyrolisis', 'H2 - SMR (Grey)']

# Use all available countries to ensure nothing is missed
if 'df_val_cap' in globals():
    all_countries = [c for c in df_val_cap['Country'].unique() if c != 'EU27']
else:
    all_countries = df_smr_updated_comp[df_smr_updated_comp['Country'] != 'EU27']['Country'].unique()

# Create a full grid of country x technology
full_grid = pd.MultiIndex.from_product([all_countries, smr_techs], names=['Country', 'Technology']).to_frame(index=False)
df_smr_updated_comp = pd.merge(full_grid, df_smr_updated_comp, on=['Country', 'Technology'], how='left').fillna(0)

# Calculate Deltas
df_smr_updated_comp['Delta (GW)'] = (df_smr_updated_comp['PyPSA Cap (GW)'] - df_smr_updated_comp['Validation Cap (GW)']).round(3)
df_smr_updated_comp['PyPSA Cap (GW)'] = df_smr_updated_comp['PyPSA Cap (GW)'].round(3)
df_smr_updated_comp['Validation Cap (GW)'] = df_smr_updated_comp['Validation Cap (GW)'].round(3)

print("\n=== UPDATED SMR (Blue & Grey): Country Level ===")
country_smr_upd = df_smr_updated_comp[df_smr_updated_comp['Country'] != 'EU27'].copy()

if not country_smr_upd.empty:
    pivot_smr_upd = country_smr_upd.pivot(index='Country', columns='Technology', values=['Validation Cap (GW)', 'PyPSA Cap (GW)', 'Delta (GW)'])
    pivot_smr_upd.columns = [f"{tech} - {metric}" for metric, tech in pivot_smr_upd.columns]
    pivot_smr_upd = pivot_smr_upd.fillna(0).sort_index()

    # Order columns nicely and force missing ones to appear
    ordered_cols = []
    for tech in smr_techs:
        for metric in ['Validation Cap (GW)', 'PyPSA Cap (GW)', 'Delta (GW)']:
            col_name = f"{tech} - {metric}"
            if col_name not in pivot_smr_upd.columns:
                pivot_smr_upd[col_name] = 0.0
            ordered_cols.append(col_name)

    pivot_smr_upd = pivot_smr_upd[ordered_cols]

    # Display countries that have at least some SMR data (either Blue or Grey)
    display_df = pivot_smr_upd[(pivot_smr_upd != 0).any(axis=1)]
    display(display_df)
else:
    print("No country-level SMR data to display.")


Re-evaluating SMR capacities in PyPSA after updates...

=== UPDATED SMR (Blue & Grey): Country Level ===


,H2 - SMR (Blue) and Pyrolisis - Validation Cap (GW),H2 - SMR (Blue) and Pyrolisis - PyPSA Cap (GW),H2 - SMR (Blue) and Pyrolisis - Delta (GW),H2 - SMR (Grey) - Validation Cap (GW),H2 - SMR (Grey) - PyPSA Cap (GW),H2 - SMR (Grey) - Delta (GW)
Country,,,,,,
BE,2.000,2.000,0.0,0.0,0.0,0.0
DE,5.306,5.306,0.0,0.0,0.0,0.0
FI,0.676,0.676,0.0,0.0,0.0,0.0
FR,0.280,0.280,0.0,0.0,0.0,0.0
IT,0.420,0.420,0.0,0.0,0.0,0.0
NL,2.400,2.400,0.0,0.0,0.0,0.0
PL,0.580,0.580,0.0,0.0,0.0,0.0
UK,5.228,5.228,0.0,0.0,0.0,0.0


In [21]:
import pandas as pd

print("=== Deep Dive: NT+ Dashboard for SMR (Grey) ===")
# Let's check the entire row for 'SMR (Grey)' in a few countries to see where the data hides
for country in ['DE', 'NL', 'UK', 'FR']:
    if country in all_sheets_data and 'Installed Capacity [GW]' in all_sheets_data[country]:
        df_cap = all_sheets_data[country]['Installed Capacity [GW]']
        smr_grey_rows = [idx for idx in df_cap.index if 'smr' in str(idx).lower() and 'grey' in str(idx).lower()]
        for idx in smr_grey_rows:
            row_data = df_cap.loc[idx]
            # Filter to show only non-null, non-zero values to find the correct column
            valid_data = row_data[pd.notna(row_data) & (row_data != '') & (row_data != 0)]
            print(f"\nCountry: {country} | Row: {idx}")
            print(valid_data)

print("\n=== Deep Dive: PyPSA Network for Gas-to-H2 Links ===")
if hasattr(n_py, 'links') and not n_py.links.empty:
    # Look for links where bus0 is a gas bus and bus1 is an h2 bus (or vice-versa)
    gas_h2_links = n_py.links[
        (n_py.links['bus0'].str.contains('gas|ch4', case=False, na=False) & n_py.links['bus1'].str.contains('h2|hydrogen', case=False, na=False)) |
        (n_py.links['bus1'].str.contains('gas|ch4', case=False, na=False) & n_py.links['bus0'].str.contains('h2|hydrogen', case=False, na=False))
    ].copy()

    print(f"Found {len(gas_h2_links)} links connecting Gas to H2.")
    if not gas_h2_links.empty:
        display(gas_h2_links[['bus0', 'bus1', 'carrier', 'p_nom', 'p_nom_opt']].head(15))
else:
    print("No links available in n_py.")


=== Deep Dive: NT+ Dashboard for SMR (Grey) ===

Country: DE | Row: H2 - SMR (Grey)
I/R           I
Excel_Row    30
Name: H2 - SMR (Grey), dtype: object

Country: NL | Row: H2 - SMR (Grey)
I/R           I
Excel_Row    30
Name: H2 - SMR (Grey), dtype: object

Country: UK | Row: H2 - SMR (Grey)
I/R           I
Excel_Row    30
Name: H2 - SMR (Grey), dtype: object

Country: FR | Row: H2 - SMR (Grey)
I/R           I
Excel_Row    30
Name: H2 - SMR (Grey), dtype: object

=== Deep Dive: PyPSA Network for Gas-to-H2 Links ===
Found 19 links connecting Gas to H2.


,bus0,bus1,carrier,p_nom,p_nom_opt
name,,,,,
SMR_b_BE00_gas_bus_BEh2,BE00_gas_bus,BEh2,natural_gas,3076.923077,3076.923077
SMR_g_DE00_gas_busDEh2Z1,DE00_gas_bus,DEh2Z1,natural_gas,6294.285714,6294.285714
SMR_b_DE00_gas_bus_DEh2,DE00_gas_bus,DEh2,natural_gas,8163.076923,8163.076923
SMR_b_FI00_gas_bus_FIh2,FI00_gas_bus,FIh2,natural_gas,1039.353846,1039.353846
SMR_b_FR00_gas_bus_FRh2,FR00_gas_bus,FRh2,natural_gas,430.769231,430.769231
SMR_b_ITN1_gas_bus_ITh2,ITN1_gas_bus,ITh2,natural_gas,646.153846,646.153846
SMR_g_LT00_gas_busLTh2Z1,LT00_gas_bus,LTh2Z1,natural_gas,2285.714286,2285.714286
SMR_b_NL00_gas_bus_NLh2,NL00_gas_bus,NLh2,natural_gas,3692.307692,3692.307692
SMR_b_PL00_gas_bus_PLh2,PL00_gas_bus,PLh2,natural_gas,892.307692,892.307692


### H2 Demand

In [28]:
import pandas as pd
import numpy as np

# Define EU27 countries for filtering
eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']

# Helper function to extract a value from a DataFrame based on Item_Key and WS037 column
def get_validation_value(data_dict, sheet, major_cat, sub_cat_key, item_key):
    value = 0.0
    if sheet in data_dict and major_cat in data_dict[sheet] and sub_cat_key in data_dict[sheet][major_cat]:
        df = data_dict[sheet][major_cat][sub_cat_key]
        if item_key in df.index:
            # Try to find the 'WS037' column, considering potential variations in column names
            val_col = [col for col in df.columns if 'WS037' in col]
            if val_col:
                extracted_val = df.loc[item_key, val_col[0]]
                print(f"    DEBUG: In get_validation_value - Sheet: {sheet}, Major Cat: {major_cat}, Sub Cat: {sub_cat_key}, Item Key: {item_key}")
                print(f"    DEBUG: Selected column: {val_col[0]}, Extracted raw value: {extracted_val}, Type: {type(extracted_val)}")
                # The value 'I' or other non-numeric strings should not be converted to float directly.
                # Only attempt conversion if it's numeric and not NaN.
                try:
                    value = float(extracted_val)
                    if pd.isna(value):
                        value = 0.0
                except (ValueError, TypeError):
                    value = 0.0
                print(f"    DEBUG: Value after conversion attempt: {value}")
    return value

print("=== Debugging Validation H2 Demand Structure ===")
# Inspect the structure for EU27 Hydrogen Consumption
if 'EU27' in validation_dataset and 'Consumption [TWh]' in validation_dataset['EU27']:
    eu27_consumption_twh = validation_dataset['EU27']['Consumption [TWh]']
    print("Keys in validation_dataset['EU27']['Consumption [TWh]']:", eu27_consumption_twh.keys())

    if 'H2' in eu27_consumption_twh: # Corrected from 'Hydrogen' to 'H2'
        h2_df = eu27_consumption_twh['H2']
        print("Index (Item_Key) for validation_dataset['EU27']['Consumption [TWh]']['H2']:", h2_df.index.tolist())
        print("Columns for validation_dataset['EU27']['Consumption [TWh]']['H2']:", h2_df.columns.tolist())
        # Explicitly check for an item key related to 'Demand' or 'Consumption' within H2
        h2_demand_keys = [k for k in h2_df.index if 'demand' in str(k).lower() or 'consumption' in str(k).lower()]
        print("Possible H2 Demand Item_Keys:", h2_demand_keys)
    else:
        print(" 'H2' key not found in validation_dataset['EU27']['Consumption [TWh]'].")
else:
    print(" 'EU27' or 'Consumption [TWh]' not found in validation_dataset.")


print("\n=== EU27 H2 Demand Comparison ===")

# 1. Validation Data for H2 Demand
# Based on the expected structure, the item_key is likely 'H2 - H2 Demand'
validation_h2_demand = get_validation_value(validation_dataset, 'EU27', 'Consumption [TWh]', 'H2', 'H2 - H2 Demand') # Corrected item_key

print(f"Validation EU27 H2 Demand: {validation_h2_demand:.2f} TWh")

# 2. PyPSA Data for H2 Demand
pypsa_h2_demand = 0.0
if hasattr(n_py, 'loads') and not n_py.loads.empty:
    # Identify H2 buses (carrier 'H2')
    h2_buses = n_py.buses[n_py.buses.carrier.str.contains('H2', case=False, na=False)].index

    # Filter loads connected to H2 buses and belonging to EU27 countries
    h2_loads_eu27 = n_py.loads[
        (n_py.loads.bus.isin(h2_buses)) &
        (n_py.loads.bus.apply(lambda x: str(x)[:2] in eu27_countries))
    ]

    if not h2_loads_eu27.empty:
        # Check if time-series data for loads is available
        if hasattr(n_py, 'loads_t') and 'p_set' in n_py.loads_t and not n_py.loads_t.p_set.empty:
            relevant_load_cols = [col for col in h2_loads_eu27.index if col in n_py.loads_t.p_set.columns]
            if relevant_load_cols:
                # Use snapshot_weightings if available, otherwise assume 1 hour per snapshot
                weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') and 'generators' in n_py.snapshot_weightings.columns else pd.Series(1.0, index=n_py.snapshots)

                # Calculate total annual demand (sum of absolute values of p_set) and convert to TWh
                pypsa_h2_demand_mwh = (n_py.loads_t.p_set[relevant_load_cols].abs().sum(axis=1) * weightings).sum()
                pypsa_h2_demand = pypsa_h2_demand_mwh / 1e6  # Convert MWh to TWh
            else:
                print("Warning: No matching time-series data found for EU27 H2 loads.")
        elif 'p_set' in h2_loads_eu27.columns: # Fallback to static p_set if time-series is not available
            hours = len(n_py.snapshots) if hasattr(n_py, 'snapshots') and len(n_py.snapshots) > 0 else 8760
            pypsa_h2_demand_mwh = h2_loads_eu27['p_set'].sum() * hours
            pypsa_h2_demand = pypsa_h2_demand_mwh / 1e6
        else:
            print("Warning: No p_set column found in PyPSA H2 loads for static calculation.")
    else:
        print("Warning: No EU27 H2 loads found in PyPSA network.")
else:
    print("Warning: No loads data found in PyPSA network for H2 demand calculation.")

print(f"PyPSA EU27 H2 Demand: {pypsa_h2_demand:.2f} TWh")

# 3. Compare H2 Demand
if validation_h2_demand != 0:
    h2_demand_delta = pypsa_h2_demand - validation_h2_demand
    h2_demand_diff_percent = (h2_demand_delta / validation_h2_demand) * 100
    print(f"Delta H2 Demand: {h2_demand_delta:.2f} TWh ({h2_demand_diff_percent:.2f} %)")
else:
    print("Cannot calculate delta or percentage for H2 demand as validation data is zero or not found.")

=== Debugging Validation H2 Demand Structure ===
Keys in validation_dataset['EU27']['Consumption [TWh]']: dict_keys(['Electricity', 'H2', 'Methane', 'CO2', 'E-Fuels'])
Index (Item_Key) for validation_dataset['EU27']['Consumption [TWh]']['H2']: ['H2 - H2 Demand', 'H2 - Shortage Quantity', 'H2 - H2 for Hydrogen GT', 'H2 - H2 for HHP Boilers', 'H2 - H2 to SNG', 'H2 - H2 to E-Liquids']
Columns for validation_dataset['EU27']['Consumption [TWh]']['H2']: ['I/R', '2030 - WS003', '2030 - WS021', '2030 - WS029', '2030 - Weighted WS', '2035 - WS032', '2035 - WS037', '2035 - WS059', '2035 - Weighted WS', '2040 - WS065', '2040 - WS071', '2040 - WS077', '2040 - Weighted WS', '2050 - WS091', '2050 - WS092', '2050 - WS106', '2050 - Weighted WS', 'Excel_Row']
Possible H2 Demand Item_Keys: ['H2 - H2 Demand']

=== EU27 H2 Demand Comparison ===
    DEBUG: In get_validation_value - Sheet: EU27, Major Cat: Consumption [TWh], Sub Cat: H2, Item Key: H2 - H2 Demand
    DEBUG: Selected column: 2035 - WS037, Ext

In [33]:
import os
import pandas as pd

# Load raw 'EU27' sheet to fetch rows 170 to 190 exactly
kpi_dashboard_path = os.path.join(VALIDATION_DIR, 'NT+_KPI_Dashboard.xlsx')
df_raw_eu27 = pd.read_excel(kpi_dashboard_path, sheet_name='EU27', header=None)

# Rows 170 to 190 in Excel correspond to index 169 to 189 in Python pandas (0-indexed)
# Column J corresponds to index 9
row_start, row_end = 169, 189
col_idx = 9  # Column J ('2035 - WS037')

# Extracted items
extracted_items = []
for r_idx in range(row_start, row_end + 1):
    label_cat = str(df_raw_eu27.iloc[r_idx, 0]).strip()
    label_sub = str(df_raw_eu27.iloc[r_idx, 1]).strip()
    val = df_raw_eu27.iloc[r_idx, col_idx]

    # Combine labels clean
    if label_cat == 'nan' or label_cat == '':
        label_cat = ""
    if label_sub == 'nan' or label_sub == '':
        label_sub = ""

    label = f"{label_cat} - {label_sub}" if label_cat and label_sub else label_sub if label_sub else label_cat
    try:
        val_float = float(val)
    except (ValueError, TypeError):
        val_float = 0.0

    extracted_items.append({
        'Excel Row': r_idx + 1,
        'Item': label,
        'Validation [TWh]': val_float
    })

df_validation_rows = pd.DataFrame(extracted_items)
display(df_validation_rows)

,Excel Row,Item,Validation [TWh]
0,170,Electricity + - Electricity Generation,4157.711850
1,171,DSR,1.842139
2,172,Net Imports,0.000000
3,173,Electricity Shortage,0.000000
4,174,Electricity - - Electricity Demand (Native + F...,3355.520625
5,175,Pump Load,145.366125
6,176,Electrolyzer Load,439.592423
7,177,Heat Electrical Usage,25.088921
8,178,Net Battery Load,14.897959
9,179,Net EV Load,273.359556


In [35]:
import numpy as np
import pandas as pd

# Define standard list of EU27 countries to aggregate PyPSA results
eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']

# Helper variables
weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') and 'generators' in n_py.snapshot_weightings.columns else 1.0
hours = len(n_py.snapshots) if hasattr(n_py, 'snapshots') else 8760

# Dictionary to save computed PyPSA metrics matching the rows
pypsa_values = {}

# 1. Electricity Generation (Row 170)
pypsa_el_gen = 0.0
if hasattr(n_py, 'generators') and not n_py.generators.empty:
    el_buses = n_py.buses[n_py.buses.carrier == 'AC'].index
    el_gens_eu27 = n_py.generators[(n_py.generators.bus.isin(el_buses)) & (n_py.generators.bus.apply(lambda x: str(x)[:2] in eu27_countries))]
    if not el_gens_eu27.empty and hasattr(n_py, 'generators_t') and 'p' in n_py.generators_t:
        cols = [col for col in el_gens_eu27.index if col in n_py.generators_t.p.columns]
        if cols:
            pypsa_el_gen += (n_py.generators_t.p[cols].sum(axis=1) * weightings).sum() / 1e6

if hasattr(n_py, 'storage_units') and not n_py.storage_units.empty:
    el_stor_eu27 = n_py.storage_units[n_py.storage_units.bus.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not el_stor_eu27.empty and hasattr(n_py, 'storage_units_t') and 'p_dispatch' in n_py.storage_units_t:
        cols = [col for col in el_stor_eu27.index if col in n_py.storage_units_t.p_dispatch.columns]
        if cols:
            pypsa_el_gen += (n_py.storage_units_t.p_dispatch[cols].sum(axis=1) * weightings).sum() / 1e6
pypsa_values['Electricity Generation'] = pypsa_el_gen

# 2. Electricity Demand (Row 174)
pypsa_el_demand = 0.0
if hasattr(n_py, 'loads') and not n_py.loads.empty:
    el_buses = n_py.buses[n_py.buses.carrier == 'AC'].index
    el_loads_eu27 = n_py.loads[(n_py.loads.bus.isin(el_buses)) & (n_py.loads.bus.apply(lambda x: str(x)[:2] in eu27_countries))]
    if not el_loads_eu27.empty and hasattr(n_py, 'loads_t') and 'p_set' in n_py.loads_t:
        cols = [col for col in el_loads_eu27.index if col in n_py.loads_t.p_set.columns]
        if cols:
            pypsa_el_demand = (n_py.loads_t.p_set[cols].sum(axis=1) * weightings).sum() / 1e6
pypsa_values['Electricity Demand'] = pypsa_el_demand

# 3. Electrolyzer Load (Row 176)
pypsa_elec_load = 0.0
if hasattr(n_py, 'links') and not n_py.links.empty:
    elec_links = n_py.links[n_py.links.index.str.contains('electrolyser|electrolyzer', case=False, na=False) & n_py.links.bus0.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not elec_links.empty and hasattr(n_py, 'links_t') and 'p0' in n_py.links_t:
        cols = [col for col in elec_links.index if col in n_py.links_t.p0.columns]
        if cols:
            pypsa_elec_load = (n_py.links_t.p0[cols].sum(axis=1) * weightings).sum() / 1e6
pypsa_values['Electrolyzer Load'] = pypsa_elec_load

# 4. H2 Production (Row 181)
pypsa_h2_gen = 0.0
if hasattr(n_py, 'links') and not n_py.links.empty:
    elec_links = n_py.links[n_py.links.index.str.contains('electrolyser|electrolyzer', case=False, na=False) & n_py.links.bus0.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not elec_links.empty and hasattr(n_py, 'links_t') and 'p1' in n_py.links_t:
        cols = [col for col in elec_links.index if col in n_py.links_t.p1.columns]
        if cols:
            pypsa_h2_gen += (n_py.links_t.p1[cols].sum(axis=1) * weightings).sum() / 1e6

if hasattr(n_py, 'links') and not n_py.links.empty:
    smr_links = n_py.links[n_py.links.carrier.isin(['SMR', 'SMR_g']) & n_py.links.bus0.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not smr_links.empty and hasattr(n_py, 'links_t') and 'p1' in n_py.links_t:
        cols = [col for col in smr_links.index if col in n_py.links_t.p1.columns]
        if cols:
            pypsa_h2_gen += (n_py.links_t.p1[cols].sum(axis=1) * weightings).sum() / 1e6
pypsa_values['H2 Domestic Production'] = pypsa_h2_gen

# 5. H2 Demand (Row 186)
pypsa_h2_demand = 0.0
if hasattr(n_py, 'loads') and not n_py.loads.empty:
    h2_buses = n_py.buses[n_py.buses.carrier.str.contains('H2', case=False, na=False)].index
    h2_loads_eu27 = n_py.loads[(n_py.loads.bus.isin(h2_buses)) & (n_py.loads.bus.apply(lambda x: str(x)[:2] in eu27_countries))]
    if not h2_loads_eu27.empty and hasattr(n_py, 'loads_t') and 'p_set' in n_py.loads_t:
        cols = [col for col in h2_loads_eu27.index if col in n_py.loads_t.p_set.columns]
        if cols:
            pypsa_h2_demand = (n_py.loads_t.p_set[cols].abs().sum(axis=1) * weightings).sum() / 1e6
pypsa_values['H2 Demand'] = pypsa_h2_demand

# 6. H2 to Hydrogen GT (Row 187)
pypsa_h2_to_gt = 0.0
if hasattr(n_py, 'links') and not n_py.links.empty:
    h2_to_power = n_py.links[n_py.links.carrier.str.contains('fuel cell|fc|h2-to-power|turbine', case=False, na=False) & n_py.links.bus0.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not h2_to_power.empty and hasattr(n_py, 'links_t') and 'p0' in n_py.links_t:
        cols = [col for col in h2_to_power.index if col in n_py.links_t.p0.columns]
        if cols:
            pypsa_h2_to_gt = (n_py.links_t.p0[cols].sum(axis=1) * weightings).sum() / 1e6
pypsa_values['H2 to Hydrogen GT'] = pypsa_h2_to_gt

def map_pypsa_to_row(row):
    item_name = str(row['Item']).lower()
    if 'generation' in item_name and 'electricity' in item_name:
        return pypsa_values.get('Electricity Generation', 0.0)
    elif 'demand' in item_name and 'electricity' in item_name:
        return pypsa_values.get('Electricity Demand', 0.0)
    elif 'electrolyzer load' in item_name:
        return pypsa_values.get('Electrolyzer Load', 0.0)
    elif 'domestic production' in item_name and 'h2' in item_name:
        return pypsa_values.get('H2 Domestic Production', 0.0)
    elif 'demand' in item_name and 'h2' in item_name:
        return pypsa_values.get('H2 Demand', 0.0)
    elif 'gt' in item_name and 'h2' in item_name:
        return pypsa_values.get('H2 to Hydrogen GT', 0.0)
    return 0.0

df_validation_rows['Validation [TWh]'] = df_validation_rows['Validation [TWh]'].round(2)
df_validation_rows['PyPSA [TWh]'] = df_validation_rows.apply(map_pypsa_to_row, axis=1).round(2)
df_validation_rows['Delta [TWh]'] = (df_validation_rows['PyPSA [TWh]'] - df_validation_rows['Validation [TWh]']).round(2)

display(df_validation_rows)

,Excel Row,Item,Validation [TWh],PyPSA [TWh],Delta [TWh]
0,170,Electricity + - Electricity Generation,4157.71,4382.01,224.30
1,171,DSR,1.84,0.00,-1.84
2,172,Net Imports,0.00,0.00,0.00
3,173,Electricity Shortage,0.00,0.00,0.00
4,174,Electricity - - Electricity Demand (Native + F...,3355.52,3539.89,184.37
5,175,Pump Load,145.37,0.00,-145.37
6,176,Electrolyzer Load,439.59,451.19,11.60
7,177,Heat Electrical Usage,25.09,0.00,-25.09
8,178,Net Battery Load,14.90,0.00,-14.90
9,179,Net EV Load,273.36,0.00,-273.36


In [41]:
import numpy as np
import pandas as pd
import os

print("=== Debugging Raw Row Labels near row 175-185 ===")
kpi_dashboard_path = os.path.join(VALIDATION_DIR, 'NT+_KPI_Dashboard.xlsx')
df_raw_eu27 = pd.read_excel(kpi_dashboard_path, sheet_name='EU27', header=None)

for idx in range(165, 191):
    if idx < len(df_raw_eu27):
        val0 = str(df_raw_eu27.iloc[idx, 0]).strip()
        val1 = str(df_raw_eu27.iloc[idx, 1]).strip()
        print(f"Row {idx+1}: Column A='{val0}' | Column B='{val1}'")

print("\n=== Detailed H2 Supply and Demand Coverage Breakdown (EU27) ===\n")

eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']
weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') and 'generators' in n_py.snapshot_weightings.columns else 1.0

# --- 1. PYPSA EXTRACTION ---
pypsa_techs = {}

# A. Electrolysis Production
if hasattr(n_py, 'links') and not n_py.links.empty:
    elec_links = n_py.links[n_py.links.index.str.contains('electrolyser|electrolyzer', case=False, na=False) & n_py.links.bus1.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not elec_links.empty and hasattr(n_py, 'links_t') and 'p1' in n_py.links_t:
        cols = [col for col in elec_links.index if col in n_py.links_t.p1.columns]
        if cols:
            pypsa_techs['Electrolysis'] = abs((n_py.links_t.p1[cols].sum(axis=1) * weightings).sum()) / 1e6

# B. SMR (carrier: SMR / SMR_g)
if hasattr(n_py, 'links') and not n_py.links.empty:
    # SMR Blue
    smr_b_links = n_py.links[n_py.links.carrier.isin(['SMR', 'natural_gas']) & n_py.links.index.str.contains('SMR_b|SMR_Blue|SMR_BE|SMR_DE|SMR_FI|SMR_FR|SMR_IT|SMR_NL|SMR_PL|SMR_UK', case=False, na=False) & n_py.links.bus1.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not smr_b_links.empty and hasattr(n_py, 'links_t') and 'p1' in n_py.links_t:
        cols = [col for col in smr_b_links.index if col in n_py.links_t.p1.columns]
        if cols:
            pypsa_techs['SMR Blue'] = abs((n_py.links_t.p1[cols].sum(axis=1) * weightings).sum()) / 1e6

    # SMR Grey
    smr_g_links = n_py.links[n_py.links.carrier.isin(['SMR_g', 'natural_gas']) & n_py.links.index.str.contains('SMR_g|SMR_Grey', case=False, na=False) & n_py.links.bus1.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not smr_g_links.empty and hasattr(n_py, 'links_t') and 'p1' in n_py.links_t:
        cols = [col for col in smr_g_links.index if col in n_py.links_t.p1.columns]
        if cols:
            pypsa_techs['SMR Grey'] = abs((n_py.links_t.p1[cols].sum(axis=1) * weightings).sum()) / 1e6

# C. H2 Storage Discharge / Charging
if hasattr(n_py, 'storage_units') and not n_py.storage_units.empty:
    h2_stores = n_py.storage_units[n_py.storage_units.carrier.str.contains('H2', case=False, na=False) & n_py.storage_units.bus.apply(lambda x: str(x)[:2] in eu27_countries)]
    if not h2_stores.empty and hasattr(n_py, 'storage_units_t'):
        if 'p_dispatch' in n_py.storage_units_t:
            cols_d = [col for col in h2_stores.index if col in n_py.storage_units_t.p_dispatch.columns]
            if cols_d:
                pypsa_techs['Storage Withdrawal'] = (n_py.storage_units_t.p_dispatch[cols_d].sum(axis=1) * weightings).sum() / 1e6
        if 'p_store' in n_py.storage_units_t:
            cols_s = [col for col in h2_stores.index if col in n_py.storage_units_t.p_store.columns]
            if cols_s:
                pypsa_techs['Storage Charging'] = (n_py.storage_units_t.p_store[cols_s].sum(axis=1) * weightings).sum() / 1e6

# D. Net Imports / Pipeline Flows
if hasattr(n_py, 'links') and not n_py.links.empty:
    h2_pipes = n_py.links[n_py.links.carrier.str.contains('H2|pipe', case=False, na=False) & n_py.links.index.str.contains('pipeline|cross_border', case=False, na=False)]
    if not h2_pipes.empty and hasattr(n_py, 'links_t') and 'p0' in n_py.links_t:
        net_import_mwh = 0.0
        for name, row in h2_pipes.iterrows():
            b0_eu = str(row['bus0'])[:2] in eu27_countries
            b1_eu = str(row['bus1'])[:2] in eu27_countries
            if b0_eu != b1_eu:
                if name in n_py.links_t.p0.columns:
                    flow = n_py.links_t.p0[name] * weightings
                    if not b0_eu and b1_eu:
                        net_import_mwh += flow.sum()
                    elif b0_eu and not b1_eu:
                        net_import_mwh -= flow.sum()
        pypsa_techs['Net Imports'] = net_import_mwh / 1e6

# Dynamic index mapping adjusted exactly to printed Row labels
row_indices = {
    'Electrolysis': 180,
    'SMR Blue': 181,
    'Net Imports': 183,
    'Storage Withdrawal': 189
}

col_idx = 9  # Column J ('2035 - WS037')
comparison_records = []

for tech, excel_idx in row_indices.items():
    label_cat = str(df_raw_eu27.iloc[excel_idx, 0]).strip() if pd.notna(df_raw_eu27.iloc[excel_idx, 0]) else ""
    label_sub = str(df_raw_eu27.iloc[excel_idx, 1]).strip() if pd.notna(df_raw_eu27.iloc[excel_idx, 1]) else ""
    if label_cat == 'nan' or label_cat == '': label_cat = ""
    if label_sub == 'nan' or label_sub == '': label_sub = ""
    label = f"{label_cat} - {label_sub}" if label_cat and label_sub else label_sub or label_cat

    raw_val = df_raw_eu27.iloc[excel_idx, col_idx]
    try:
        val_float = float(raw_val)
        if pd.isna(val_float): val_float = 0.0
    except (ValueError, TypeError):
        val_float = 0.0

    pypsa_val = pypsa_techs.get(tech, 0.0)

    comparison_records.append({
        'Technology / Source': tech,
        'Dashboard Item': label,
        'Validation [TWh]': round(val_float, 2),
        'PyPSA [TWh]': round(pypsa_val, 2),
        'Delta [TWh]': round(pypsa_val - val_float, 2)
    })

# Manual addition of Storage Charging as it shares the net storage cell
pypsa_charge = pypsa_techs.get('Storage Charging', 0.0)
comparison_records.append({
    'Technology / Source': 'Storage Charging',
    'Dashboard Item': 'Net H2 Storage Load (Charging)',
    'Validation [TWh]': 0.0,
    'PyPSA [TWh]': round(pypsa_charge, 2),
    'Delta [TWh]': round(pypsa_charge, 2)
})

df_comparison = pd.DataFrame(comparison_records)
display(df_comparison)

=== Debugging Raw Row Labels near row 175-185 ===
Row 166: Column A='Import Share [%]' | Column B='nan'
Row 167: Column A='H2' | Column B='Share of Domestic Production (Supply %)'
Row 168: Column A='nan' | Column B='Share of Non-EU Imports (Supply %)'
Row 169: Column A='Energy Balance [TWh]' | Column B='nan'
Row 170: Column A='Electricity +' | Column B='Electricity Generation'
Row 171: Column A='nan' | Column B='DSR'
Row 172: Column A='nan' | Column B='Net Imports'
Row 173: Column A='nan' | Column B='Electricity Shortage'
Row 174: Column A='Electricity -' | Column B='Electricity Demand (Native + Fixed)'
Row 175: Column A='nan' | Column B='Pump Load'
Row 176: Column A='nan' | Column B='Electrolyzer Load'
Row 177: Column A='nan' | Column B='Heat Electrical Usage'
Row 178: Column A='nan' | Column B='Net Battery Load'
Row 179: Column A='nan' | Column B='Net EV Load'
Row 180: Column A='Electricity Balance' | Column B='nan'
Row 181: Column A='H2 +' | Column B='H2 Domestic Production'
Row 182

,Technology / Source,Dashboard Item,Validation [TWh],PyPSA [TWh],Delta [TWh]
0,Electrolysis,H2 + - H2 Domestic Production,523.50,441.63,-81.87
1,SMR Blue,NH3 Imports,58.30,0.17,-58.13
2,Net Imports,Net Imports,0.00,0.00,0.00
3,Storage Withdrawal,Net H2 Storage Load,0.72,68.44,67.71
4,Storage Charging,Net H2 Storage Load (Charging),0.00,69.13,69.13


In [42]:
import os
import pandas as pd
import numpy as np

# 1. Load raw 'EU27' sheet to fetch rows 181 to 185 (0-indexed 180 to 184)
kpi_dashboard_path = os.path.join(VALIDATION_DIR, 'NT+_KPI_Dashboard.xlsx')
df_raw_eu27 = pd.read_excel(kpi_dashboard_path, sheet_name='EU27', header=None)

row_start, row_end = 180, 184  # Excel Rows 181-185 correspond to Python indices 180-184
col_idx = 9  # Column J ('2035 - WS037')

validation_imports = {}
for r_idx in range(row_start, row_end + 1):
    label_cat = str(df_raw_eu27.iloc[r_idx, 0]).strip()
    label_sub = str(df_raw_eu27.iloc[r_idx, 1]).strip()

    if label_cat in ['nan', '']: label_cat = ""
    if label_sub in ['nan', '']: label_sub = ""
    label = f"{label_cat} - {label_sub}" if label_cat and label_sub else label_sub or label_cat

    raw_val = df_raw_eu27.iloc[r_idx, col_idx]
    try:
        val_float = float(raw_val)
        if pd.isna(val_float): val_float = 0.0
    except (ValueError, TypeError):
        val_float = 0.0

    validation_imports[label] = val_float

# 2. Extract corresponding Import metrics from PyPSA
eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']
weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') and 'generators' in n_py.snapshot_weightings.columns else 1.0

pypsa_imports = {
    'NH3 Imports': 0.0,
    'Non-EU Imports': 0.0,
    'Net Imports': 0.0
}

# Extract Non-EU Imports (from H2 Import generators)
if hasattr(n_py, 'generators') and not n_py.generators.empty:
    h2_imp_gens = n_py.generators[
        n_py.generators.carrier.str.contains('H2_import|import', case=False, na=False) &
        n_py.generators.bus.apply(lambda x: str(x)[:2] in eu27_countries)
    ]
    if not h2_imp_gens.empty and hasattr(n_py, 'generators_t') and 'p' in n_py.generators_t:
        cols = [col for col in h2_imp_gens.index if col in n_py.generators_t.p.columns]
        if cols:
            pypsa_imports['Non-EU Imports'] = (n_py.generators_t.p[cols].sum(axis=1) * weightings).sum() / 1e6

# Extract Net Inter-regional Pipeline Flows (Net Imports) across EU27 boundary
if hasattr(n_py, 'links') and not n_py.links.empty:
    h2_pipes = n_py.links[
        n_py.links.carrier.str.contains('H2', case=False, na=False) &
        n_py.links.index.str.contains('pipeline|cross_border', case=False, na=False)
    ]
    if not h2_pipes.empty and hasattr(n_py, 'links_t') and 'p0' in n_py.links_t:
        net_import_mwh = 0.0
        for name, row in h2_pipes.iterrows():
            b0_eu = str(row['bus0'])[:2] in eu27_countries
            b1_eu = str(row['bus1'])[:2] in eu27_countries
            if b0_eu != b1_eu:
                if name in n_py.links_t.p0.columns:
                    flow = n_py.links_t.p0[name] * weightings
                    if not b0_eu and b1_eu:
                        net_import_mwh += flow.sum()
                    elif b0_eu and not b1_eu:
                        net_import_mwh -= flow.sum()
        pypsa_imports['Net Imports'] = net_import_mwh / 1e6

# 3. Create Comparison DataFrame
comp_data = []
mapping = {
    'nan - NH3 Imports': 'NH3 Imports',
    'nan - Non-EU Imports': 'Non-EU Imports',
    'nan - Net Imports': 'Net Imports'
}

for raw_label, val_val in validation_imports.items():
    py_key = mapping.get(raw_label, None)
    py_val = pypsa_imports.get(py_key, 0.0) if py_key else 0.0

    comp_data.append({
        'Excel Label': raw_label,
        'Validation [TWh]': round(val_val, 2),
        'PyPSA [TWh]': round(py_val, 2),
        'Delta [TWh]': round(py_val - val_val, 2)
    })

df_import_comparison = pd.DataFrame(comp_data)
display(df_import_comparison)

,Excel Label,Validation [TWh],PyPSA [TWh],Delta [TWh]
0,H2 + - H2 Domestic Production,523.50,0.0,-523.50
1,NH3 Imports,58.30,0.0,-58.30
2,Non-EU Imports,55.47,0.0,-55.47
3,Net Imports,0.00,0.0,0.00
4,H2 Shortage,0.00,0.0,0.00


In [43]:
print("=== Debugging PyPSA H2 Network Carriers and Elements ===")
if hasattr(n_py, 'generators') and not n_py.generators.empty:
    print("\nUnique generator carriers:", n_py.generators.carrier.unique())
    h2_gens = n_py.generators[n_py.generators.carrier.str.contains('H2|import|nh3', case=False, na=False)]
    if not h2_gens.empty:
        print("\nH2/Import generators found:")
        print(h2_gens[['bus', 'carrier', 'p_nom', 'p_nom_opt']].head(10))

if hasattr(n_py, 'links') and not n_py.links.empty:
    print("\nUnique link carriers:", n_py.links.carrier.unique())
    h2_links = n_py.links[n_py.links.carrier.str.contains('H2|import|nh3|pipeline', case=False, na=False)]
    if not h2_links.empty:
        print("\nH2/Import links found (top 10):")
        print(h2_links[['bus0', 'bus1', 'carrier', 'p_nom', 'p_nom_opt']].head(10))

=== Debugging PyPSA H2 Network Carriers and Elements ===

Unique generator carriers: ['hard_coal' 'lignite' 'oil' 'natural_gas' 'AC' 'other_non_res' 'nuclear'
 'solar' 'wind_onshore' 'hydro' 'RES' 'wind_offshore' 'solar_thermal' 'H2'
 'import' 'H2_import']

H2/Import generators found:
                    bus carrier  p_nom  p_nom_opt
name                                             
Slack_H2_ATh2      ATh2      H2    0.0    0.00000
Slack_H2_BEh2      BEh2      H2    0.0    0.00000
Slack_H2_BGh2      BGh2      H2    0.0    0.00000
Slack_H2_CHh2      CHh2      H2    0.0    0.00000
Slack_H2_CYh2      CYh2      H2    0.0    0.00000
Slack_H2_CZh2      CZh2      H2    0.0    0.00000
Slack_H2_DEh2      DEh2      H2    0.0    0.00000
Slack_H2_DEh2Z1  DEh2Z1      H2    0.0    0.00000
Slack_H2_DKh2      DKh2      H2    0.0    0.00000
Slack_H2_EEh2      EEh2      H2    0.0  133.03766

Unique link carriers: ['natural_gas' 'oil' 'lignite' 'hard_coal' 'AC' 'H2' 'cross_border' 'SMR']

H2/Import links

In [45]:
import numpy as np
import pandas as pd

# Define EU27 countries
eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']
weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') and 'generators' in n_py.snapshot_weightings.columns else 1.0

pypsa_imports = {
    'NH3 Imports': 0.0,
    'Non-EU Imports': 0.0,
    'Net Imports': 0.0
}

# 1. Non-EU Imports (from H2_import and import generators on H2/AC buses)
if hasattr(n_py, 'generators') and not n_py.generators.empty:
    import_gens = n_py.generators[
        n_py.generators.carrier.isin(['H2_import', 'import']) &
        n_py.generators.bus.apply(lambda x: str(x)[:2] in eu27_countries)
    ]
    if not import_gens.empty and hasattr(n_py, 'generators_t') and 'p' in n_py.generators_t:
        cols = [col for col in import_gens.index if col in n_py.generators_t.p.columns]
        if cols:
            total_imports = (n_py.generators_t.p[cols].sum(axis=1) * weightings).sum() / 1e6
            # Distribute dynamically based on name clues or assign to Non-EU
            nh3_cols = [c for c in cols if 'nh3' in str(c).lower()]
            if nh3_cols:
                pypsa_imports['NH3 Imports'] = (n_py.generators_t.p[nh3_cols].sum(axis=1) * weightings).sum() / 1e6
                pypsa_imports['Non-EU Imports'] = total_imports - pypsa_imports['NH3 Imports']
            else:
                pypsa_imports['Non-EU Imports'] = total_imports

# 2. Net Imports (Inter-regional pipeline links across the EU27 borders)
if hasattr(n_py, 'links') and not n_py.links.empty:
    border_links = n_py.links[
        n_py.links.carrier.isin(['cross_border', 'H2']) &
        (n_py.links.index.str.contains('pipeline|pipe|export|import', case=False, na=False) | (n_py.links.carrier == 'cross_border'))
    ]
    if not border_links.empty and hasattr(n_py, 'links_t') and 'p0' in n_py.links_t:
        net_import_mwh = 0.0
        for name, row in border_links.iterrows():
            b0_eu = str(row['bus0'])[:2] in eu27_countries
            b1_eu = str(row['bus1'])[:2] in eu27_countries
            if b0_eu != b1_eu:
                if name in n_py.links_t.p0.columns:
                    flow = n_py.links_t.p0[name] * weightings
                    # Flow from outside to inside EU27 is positive net import
                    if not b0_eu and b1_eu:
                        net_import_mwh += flow.sum()
                    elif b0_eu and not b1_eu:
                        net_import_mwh -= flow.sum()
        pypsa_imports['Net Imports'] = net_import_mwh / 1e6

# Compare with validation_imports
comp_data = []
mapping = {
    'nan - NH3 Imports': 'NH3 Imports',
    'nan - Non-EU Imports': 'Non-EU Imports',
    'nan - Net Imports': 'Net Imports'
}

for raw_label, val_val in validation_imports.items():
    py_key = mapping.get(raw_label, None)
    py_val = pypsa_imports.get(py_key, 0.0) if py_key else 0.0
    comp_data.append({
        'Excel Label': raw_label,
        'Validation [TWh]': round(val_val, 2),
        'PyPSA [TWh]': round(py_val, 2),
        'Delta [TWh]': round(py_val - val_val, 2)
    })

df_import_comparison = pd.DataFrame(comp_data)
display(df_import_comparison)

,Excel Label,Validation [TWh],PyPSA [TWh],Delta [TWh]
0,H2 + - H2 Domestic Production,523.50,0.0,-523.50
1,NH3 Imports,58.30,0.0,-58.30
2,Non-EU Imports,55.47,0.0,-55.47
3,Net Imports,0.00,0.0,0.00
4,H2 Shortage,0.00,0.0,0.00


### Electricity Installed Power

In [22]:
import pandas as pd
import numpy as np

print("Extracting Electricity Installed Capacity [GW]...")

pypsa_el_records = []

# 1. PyPSA Generators
if hasattr(n_py, 'generators') and not n_py.generators.empty:
    for idx, row in n_py.generators.iterrows():
        country = str(row['bus'])[:2]
        carrier = str(row['carrier'])
        cap_gw = row.get('p_nom_opt', row.get('p_nom', 0)) / 1000.0

        tech_map = {
            'wind_onshore': 'Electricity - Wind Onshore',
            'wind_offshore': 'Electricity - Wind Offshore',
            'solar': 'Electricity - Solar PV',
            'CSP': 'Electricity - Solar CSP',
            'nuclear': 'Electricity - Nuclear',
            'natural_gas': 'Electricity - Natural Gas',
            'hard_coal': 'Electricity - Hard Coal',
            'lignite': 'Electricity - Lignite',
            'oil': 'Electricity - Oil',
            'other_non_res': 'Electricity - Other Non-RES',
            'hydro': 'Electricity - Hydro (RoR + Pondage + Reservoir)',
            'RES': 'Electricity - Other RES'
        }

        mapped_tech = tech_map.get(carrier, f"Electricity - Other ({carrier})")
        if carrier not in ['H2', 'H2_import', 'AC', 'heat', 'import']:
            pypsa_el_records.append({'Country': country, 'Technology': mapped_tech, 'PyPSA Cap (GW)': cap_gw})

# 2. PyPSA Storage Units (Batteries, Pumped Hydro)
if hasattr(n_py, 'storage_units') and not n_py.storage_units.empty:
    for idx, row in n_py.storage_units.iterrows():
        country = str(row['bus'])[:2]
        carrier = str(row['carrier'])
        cap_gw = row.get('p_nom_opt', row.get('p_nom', 0)) / 1000.0

        tech_map = {
            'battery': 'Electricity - Batteries',
            'PHS': 'Electricity - Storage (hydro)'
        }
        mapped_tech = tech_map.get(carrier, f"Electricity - Storage ({carrier})")
        pypsa_el_records.append({'Country': country, 'Technology': mapped_tech, 'PyPSA Cap (GW)': cap_gw})

df_pypsa_el = pd.DataFrame(pypsa_el_records)
if not df_pypsa_el.empty:
    df_pypsa_el = df_pypsa_el.groupby(['Country', 'Technology'], as_index=False)['PyPSA Cap (GW)'].sum()
else:
    df_pypsa_el = pd.DataFrame(columns=['Country', 'Technology', 'PyPSA Cap (GW)'])

# 3. Validation Data Extraction (Robust raw extraction)
val_el_records = []

for sheet in country_sheets:
    try:
        temp_df = pd.read_excel(xl, sheet_name=sheet, header=None)
        found_coords = np.where(temp_df.isin(['WS037']))

        if len(found_coords[0]) > 0:
            col_idx = found_coords[1][0]

            current_category = None
            for row_idx in range(4, 30):
                if row_idx < len(temp_df):
                    label_col0 = str(temp_df.iloc[row_idx, 0]).strip()
                    label_col1 = str(temp_df.iloc[row_idx, 1]).strip()

                    if label_col0.lower() != 'nan' and label_col0 != '':
                        current_category = label_col0

                    label = label_col1 if label_col1.lower() != 'nan' else ""

                    if current_category == 'Electricity' and label:
                        val = temp_df.iloc[row_idx, col_idx]
                        try:
                            val = float(val)
                            if not np.isnan(val) and val != 0:
                                tech = f"Electricity - {label}"

                                # Map to grouped categories
                                if 'Wind Offshore' in label:
                                    tech = 'Electricity - Wind Offshore'
                                elif 'Run of River' in label or 'Pondage' in label or 'Reservoir' in label:
                                    tech = 'Electricity - Hydro (RoR + Pondage + Reservoir)'
                                elif 'Batteries' in label and 'GWh' not in label:
                                    tech = 'Electricity - Batteries'
                                elif 'Batteries' in label and 'GWh' in label:
                                    continue
                                elif 'PS Turbine' in label or 'PS Pump' in label:
                                    tech = 'Electricity - Storage (hydro)'

                                val_el_records.append({'Country': sheet, 'Technology': tech, 'Validation Cap (GW)': val})
                        except (ValueError, TypeError):
                            continue
    except Exception as e:
        continue

df_val_el = pd.DataFrame(val_el_records)
if not df_val_el.empty:
    df_val_el = df_val_el.groupby(['Country', 'Technology'], as_index=False)['Validation Cap (GW)'].sum()
else:
    df_val_el = pd.DataFrame(columns=['Country', 'Technology', 'Validation Cap (GW)'])

# 4. Merge & Compare
if not df_pypsa_el.empty or not df_val_el.empty:
    df_el_comp = pd.merge(df_val_el, df_pypsa_el, on=['Country', 'Technology'], how='outer')
    df_el_comp['PyPSA Cap (GW)'] = df_el_comp['PyPSA Cap (GW)'].fillna(0)
    df_el_comp['Validation Cap (GW)'] = df_el_comp['Validation Cap (GW)'].fillna(0)

    df_el_comp['Delta (GW)'] = (df_el_comp['PyPSA Cap (GW)'] - df_el_comp['Validation Cap (GW)']).round(3)
    df_el_comp['PyPSA Cap (GW)'] = df_el_comp['PyPSA Cap (GW)'].round(3)
    df_el_comp['Validation Cap (GW)'] = df_el_comp['Validation Cap (GW)'].round(3)

    df_el_comp = df_el_comp[(df_el_comp['PyPSA Cap (GW)'] != 0) | (df_el_comp['Validation Cap (GW)'] != 0)]

    eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']
    df_el_eu27 = df_el_comp[df_el_comp['Country'].isin(eu27_countries)].groupby('Technology', as_index=False)[['Validation Cap (GW)', 'PyPSA Cap (GW)']].sum()
    df_el_eu27['Country'] = 'EU27'
    df_el_eu27['Delta (GW)'] = (df_el_eu27['PyPSA Cap (GW)'] - df_el_eu27['Validation Cap (GW)']).round(3)

    print("\n=== Electricity Installed Capacity [GW]: EU27 Aggregated ===")
    display(df_el_eu27.sort_values('PyPSA Cap (GW)', ascending=False).set_index('Technology').drop(columns=['Country']))

    print("\n=== Electricity Installed Capacity [GW]: Key Countries (DE, FR, NL, UK) ===")
    sample_countries = df_el_comp[df_el_comp['Country'].isin(['DE', 'FR', 'NL', 'UK'])]
    if not sample_countries.empty:
        pivot_el = sample_countries.pivot(index='Technology', columns='Country', values=['Validation Cap (GW)', 'PyPSA Cap (GW)', 'Delta (GW)'])
        pivot_el = pivot_el.fillna(0)
        display(pivot_el)
else:
    print("No electricity capacity data found to compare.")


Extracting Electricity Installed Capacity [GW]...

=== Electricity Installed Capacity [GW]: EU27 Aggregated ===


,Validation Cap (GW),PyPSA Cap (GW),Delta (GW)
Technology,,,
Electricity - Solar PV,897.135,899.212,2.077
Electricity - Wind Onshore,455.788,455.789,0.001
Electricity - Wind Offshore,158.900,157.224,-1.676
Electricity - Storage (hydro),113.330,128.502,15.172
Electricity - Batteries,144.979,119.663,-25.316
Electricity - Nuclear,97.293,97.293,0.000
Electricity - Storage (H2),0.000,48.512,48.512
Electricity - Hydro (RoR + Pondage + Reservoir),109.095,41.421,-67.674
Electricity - Other RES,33.453,33.453,0.000



=== Electricity Installed Capacity [GW]: Key Countries (DE, FR, NL, UK) ===


Validation Cap (GW)          \
Country                                                          DE      FR   
Technology                                                                    
Electricity - Batteries                                      39.553   0.000   
Electricity - Biofuel                                         0.000   0.000   
Electricity - Coal                                            4.782   0.000   
Electricity - Crude Oil                                       0.322   1.331   
Electricity - Hydro (RoR + Pondage + Reservoir)               4.942  21.449   
Electricity - Hydrogen GT                                    16.537   0.000   
Electricity - Natural Gas                                    32.721   7.189   
Electricity - Nuclear                                         0.000  63.020   
Electricity - Other Non RES                                   2.938   2.842   
Electricity - Other Non-RES                                   0.000   0.000   
Electricity - Other RES                                      10.438   1.939   
Electricity - SRES Electricity                                1.357   0.000   
Electricity - Solar PV                                      308.587  72.080   
Electricity - Storage (H2)                                    0.000   0.000   
Electricity - Storage (hydro)                                18.702   8.400   
Electricity - Wind Offshore                                  50.425  10.775   
Electricity - Wind Onshore                                  156.055  37.580   

                                                                 \
Country                                              NL      UK   
Technology                                                        
Electricity - Batteries                          21.744  32.821   
Electricity - Biofuel                             2.211   0.000   
Electricity - Coal                                0.000   0.000   
Electricity - Crude Oil                           0.000   0.000   
Electricity - Hydro (RoR + Pondage + Reservoir)   0.037   2.009   
Electricity - Hydrogen GT                         3.072   3.887   
Electricity - Natural Gas                        12.725   8.738   
Electricity - Nuclear                             0.486   5.000   
Electricity - Other Non RES                       2.329   0.000   
Electricity - Other Non-RES                       0.000   0.000   
Electricity - Other RES                           1.204   8.657   
Electricity - SRES Electricity                    0.000   0.000   
Electricity - Solar PV                           70.938  69.121   
Electricity - Storage (H2)                        0.000   0.000   
Electricity - Storage (hydro)                     0.000   8.232   
Electricity - Wind Offshore                      27.524  88.553   
Electricity - Wind Onshore                        9.330  31.178   

                                                PyPSA Cap (GW)          \
Country                                                     DE      FR   
Technology                                                               
Electricity - Batteries                                 28.076   0.000   
Electricity - Biofuel                                    0.000   0.000   
Electricity - Coal                                       0.000   0.000   
Electricity - Crude Oil                                  0.000   0.000   
Electricity - Hydro (RoR + Pondage + Reservoir)          3.934  11.600   
Electricity - Hydrogen GT                                0.000   0.000   
Electricity - Natural Gas                                0.000   0.000   
Electricity - Nuclear                                    0.000  63.020   
Electricity - Other Non RES                              0.000   0.000   
Electricity - Other Non-RES                              2.938   2.842   
Electricity - Other RES                                 10.438   1.939   
Electricity - SRES Electricity                           0.000   0.000   
Electricity - So

In [23]:
import pandas as pd
import numpy as np

print("=== Detailed Extraction: Wind Offshore to Batteries [GWh] ===")

# 1. Validation Data Extraction (Detailed raw extraction)
detail_val_records = []

for sheet in country_sheets:
    try:
        temp_df = pd.read_excel(xl, sheet_name=sheet, header=None)
        found_coords = np.where(temp_df.isin(['WS037']))

        if len(found_coords[0]) > 0:
            col_idx = found_coords[1][0]

            current_category = None
            # Rows 5 to 29 in Excel cover from Wind Onshore down to Batteries [GWh]
            for row_idx in range(5, 30):
                if row_idx < len(temp_df):
                    label_col0 = str(temp_df.iloc[row_idx, 0]).strip()
                    label_col1 = str(temp_df.iloc[row_idx, 1]).strip()

                    if label_col0.lower() != 'nan' and label_col0 != '':
                        current_category = label_col0

                    label = label_col1 if label_col1.lower() != 'nan' else ""

                    if current_category == 'Electricity' and label:
                        val = temp_df.iloc[row_idx, col_idx]
                        try:
                            val = float(val)
                            if not np.isnan(val) and val != 0:
                                tech = f"Electricity - {label}"
                                detail_val_records.append({'Country': sheet, 'Technology': tech, 'Validation Cap': val})
                        except (ValueError, TypeError):
                            continue
    except Exception as e:
        continue

df_detail_val = pd.DataFrame(detail_val_records)
if not df_detail_val.empty:
    df_detail_val = df_detail_val.groupby(['Country', 'Technology'], as_index=False)['Validation Cap'].sum()
else:
    df_detail_val = pd.DataFrame(columns=['Country', 'Technology', 'Validation Cap'])

# 2. PyPSA Detailed Mapping (Best effort for 1-to-1 match)
detail_pypsa_records = []

if hasattr(n_py, 'generators') and not n_py.generators.empty:
    for idx, row in n_py.generators.iterrows():
        country = str(row['bus'])[:2]
        carrier = str(row['carrier'])
        cap = row.get('p_nom_opt', row.get('p_nom', 0)) / 1000.0

        tech_map = {
            'wind_offshore': 'Electricity - Wind Offshore Radial', # Assuming radial as default
            'solar': 'Electricity - Solar PV',
            'CSP': 'Electricity - Solar CSP',
            'nuclear': 'Electricity - Nuclear',
            'natural_gas': 'Electricity - Natural Gas',
            'hard_coal': 'Electricity - Hard Coal',
            'lignite': 'Electricity - Lignite',
            'oil': 'Electricity - Crude Oil',
            'hydro': 'Electricity - Run of River', # Rough assignment for detailed view
            'RES': 'Electricity - Other RES',
            'other_non_res': 'Electricity - Other Non-RES'
        }

        if carrier in tech_map:
            detail_pypsa_records.append({'Country': country, 'Technology': tech_map[carrier], 'PyPSA Cap': cap})

if hasattr(n_py, 'storage_units') and not n_py.storage_units.empty:
    for idx, row in n_py.storage_units.iterrows():
        country = str(row['bus'])[:2]
        carrier = str(row['carrier'])
        cap_gw = row.get('p_nom_opt', row.get('p_nom', 0)) / 1000.0
        cap_gwh = cap_gw * row.get('max_hours', 1)  # Approximate GWh

        if carrier == 'battery':
            detail_pypsa_records.append({'Country': country, 'Technology': 'Electricity - Batteries', 'PyPSA Cap': cap_gw})
            detail_pypsa_records.append({'Country': country, 'Technology': 'Electricity - Batteries [GWh]', 'PyPSA Cap': cap_gwh})
        elif carrier == 'PHS':
            detail_pypsa_records.append({'Country': country, 'Technology': 'Electricity - PS Turbine', 'PyPSA Cap': cap_gw})

df_detail_pypsa = pd.DataFrame(detail_pypsa_records)
if not df_detail_pypsa.empty:
    df_detail_pypsa = df_detail_pypsa.groupby(['Country', 'Technology'], as_index=False)['PyPSA Cap'].sum()
else:
    df_detail_pypsa = pd.DataFrame(columns=['Country', 'Technology', 'PyPSA Cap'])

# 3. Merge & Display EU27
if not df_detail_val.empty:
    df_detail_comp = pd.merge(df_detail_val, df_detail_pypsa, on=['Country', 'Technology'], how='outer').fillna(0)
    df_detail_comp['Delta'] = (df_detail_comp['PyPSA Cap'] - df_detail_comp['Validation Cap']).round(3)

    # EU27 Aggregation
    eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']
    df_detail_eu27 = df_detail_comp[df_detail_comp['Country'].isin(eu27_countries)].groupby('Technology', as_index=False)[['Validation Cap', 'PyPSA Cap']].sum()
    df_detail_eu27['Delta'] = (df_detail_eu27['PyPSA Cap'] - df_detail_eu27['Validation Cap']).round(3)

    # Filter to requested range of rows (excluding Wind Onshore since user said "Wind Offshore to Batteries [GWh]")
    df_detail_eu27 = df_detail_eu27[df_detail_eu27['Technology'] != 'Electricity - Wind Onshore']

    display(df_detail_eu27.set_index('Technology'))
else:
    print("No detailed validation data found.")

=== Detailed Extraction: Wind Offshore to Batteries [GWh] ===


,Validation Cap,PyPSA Cap,Delta
Technology,,,
Electricity - Adequacy Units,15.000000,0.000000,-15.000
Electricity - Batteries,144.979400,119.663379,-25.316
Electricity - Batteries [GWh],621.072200,562.059958,-59.012
Electricity - Biofuel,6.810449,0.000000,-6.810
Electricity - Coal,15.907071,0.000000,-15.907
Electricity - Crude Oil,3.811209,0.000000,-3.811
Electricity - Hydrogen GT,20.423596,0.000000,-20.424
Electricity - Natural Gas,176.560958,0.000000,-176.561
Electricity - Nuclear,97.292990,97.292993,0.000


In [24]:
import pandas as pd
import numpy as np

# Define EU27 countries for filtering
eu27_countries = ['AT', 'BE', 'BG', 'HR', 'CY', 'CZ', 'DK', 'EE', 'FI', 'FR', 'DE', 'GR', 'HU', 'IE', 'IT', 'LV', 'LT', 'LU', 'MT', 'NL', 'PL', 'PT', 'RO', 'SK', 'SI', 'ES', 'SE']

# Helper function to extract a value from a DataFrame based on Item_Key and WS037 column
def get_validation_value(data_dict, major_cat, sub_cat_key, item_key):
    value = 0.0
    if 'EU27' in data_dict and major_cat in data_dict['EU27'] and sub_cat_key in data_dict['EU27'][major_cat]:
        df = data_dict['EU27'][major_cat][sub_cat_key]
        if item_key in df.index:
            val_col = [col for col in df.columns if 'WS037' in col]
            if val_col:
                extracted_val = df.loc[item_key, val_col[0]]
                value = float(extracted_val) if pd.notna(extracted_val) and pd.api.types.is_numeric(extracted_val) else 0.0
    return value

# Helper function to sum generation values from a DataFrame based on WS037 column
def get_validation_generation_sum(data_dict, major_cat, sub_cat_key):
    total_generation = 0.0
    if 'EU27' in data_dict and major_cat in data_dict['EU27'] and sub_cat_key in data_dict['EU27'][major_cat]:
        df = data_dict['EU27'][major_cat][sub_cat_key]
        val_col = [col for col in df.columns if 'WS037' in col]
        if val_col:
            # Sum all numeric values in the WS037 column for the 'Electricity' sub-category
            relevant_vals = pd.to_numeric(df[val_col[0]], errors='coerce').fillna(0)
            total_generation = relevant_vals.sum()
    return total_generation

print("=== EU27 Electricity Demand Comparison ===")

# 1. Validation Data for Electricity Demand
# Assuming 'pypsa_dataset' is the dictionary containing the loaded excel data.
# The path to 'Electricity - Demand' is within pypsa_dataset['EU27']['Consumption [TWh]']['Electricity']
validation_el_demand = get_validation_value(pypsa_dataset, 'Consumption [TWh]', 'Electricity', 'Electricity - Demand')

print(f"Validation EU27 Electricity Demand: {validation_el_demand:.2f} TWh")

# 2. PyPSA Data for Electricity Demand
pypsa_el_demand = 0.0
if hasattr(n_py, 'loads_t') and 'p_set' in n_py.loads_t and not n_py.loads_t.p_set.empty:
    # Identify electricity buses (carrier 'AC')
    electricity_buses = n_py.buses[n_py.buses.carrier == 'AC'].index

    # Filter loads connected to electricity buses and belonging to EU27 countries
    el_loads_eu27 = n_py.loads[
        (n_py.loads.bus.isin(electricity_buses)) &
        (n_py.loads.bus.apply(lambda x: str(x)[:2] in eu27_countries))
    ]

    if not el_loads_eu27.empty:
        # Get time-series data for these loads
        relevant_load_cols = [col for col in el_loads_eu27.index if col in n_py.loads_t.p_set.columns]
        if relevant_load_cols:
            # Use snapshot_weightings if available, otherwise assume 1 hour per snapshot
            weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') and 'generators' in n_py.snapshot_weightings.columns else pd.Series(1.0, index=n_py.snapshots)

            # Calculate total annual demand (sum of absolute values of p_set)
            pypsa_el_demand_mwh = (n_py.loads_t.p_set[relevant_load_cols].abs().sum(axis=1) * weightings).sum()
            pypsa_el_demand = pypsa_el_demand_mwh / 1e6  # Convert MWh to TWh
        else:
            print("Warning: No matching time-series data found for EU27 electricity loads.")
    else:
        print("Warning: No EU27 electricity loads found in PyPSA network.")
elif hasattr(n_py, 'loads') and not n_py.loads.empty:
    # Fallback for static loads if time-series not available or empty
    electricity_buses = n_py.buses[n_py.buses.carrier == 'AC'].index
    el_loads_eu27 = n_py.loads[
        (n_py.loads.bus.isin(electricity_buses)) &
        (n_py.loads.bus.apply(lambda x: str(x)[:2] in eu27_countries))
    ]
    if not el_loads_eu27.empty:
        hours = len(n_py.snapshots) if hasattr(n_py, 'snapshots') and len(n_py.snapshots) > 0 else 8760
        pypsa_el_demand_mwh = el_loads_eu27['p_set'].sum() * hours
        pypsa_el_demand = pypsa_el_demand_mwh / 1e6
    else:
        print("Warning: No static EU27 electricity loads found in PyPSA network.")
else:
    print("Warning: No loads data found in PyPSA network for demand calculation.")

print(f"PyPSA EU27 Electricity Demand: {pypsa_el_demand:.2f} TWh")

# 3. Compare Electricity Demand
if validation_el_demand != 0:
    demand_delta = pypsa_el_demand - validation_el_demand
    demand_diff_percent = (demand_delta / validation_el_demand) * 100
    print(f"Delta Electricity Demand: {demand_delta:.2f} TWh ({demand_diff_percent:.2f} %)")
else:
    print("Cannot calculate delta or percentage for electricity demand as validation data is zero.")

print("\n=== EU27 Annual Electricity Generation Comparison ===")

# 1. Validation Data for Electricity Generation
# The path to 'Electricity' generation is within pypsa_dataset['EU27']['Generation [TWh]']['Electricity']
validation_el_generation = get_validation_generation_sum(pypsa_dataset, 'Generation [TWh]', 'Electricity')

print(f"Validation EU27 Annual Electricity Generation: {validation_el_generation:.2f} TWh")

# 2. PyPSA Data for Electricity Generation
pypsa_el_generation = 0.0
if (hasattr(n_py, 'generators_t') and 'p' in n_py.generators_t and not n_py.generators_t.p.empty) or \
   (hasattr(n_py, 'storage_units_t') and 'p_dispatch' in n_py.storage_units_t and not n_py.storage_units_t.p_dispatch.empty):

    # Identify electricity buses (carrier 'AC')
    electricity_buses = n_py.buses[n_py.buses.carrier == 'AC'].index

    # Use snapshot_weightings if available
    weightings = n_py.snapshot_weightings.generators if hasattr(n_py, 'snapshot_weightings') and 'generators' in n_py.snapshot_weightings.columns else pd.Series(1.0, index=n_py.snapshots)

    # Generators contributing to electricity generation
    if hasattr(n_py, 'generators') and not n_py.generators.empty:
        # Filter generators on electricity buses and in EU27 countries
        el_generators_eu27 = n_py.generators[
            (n_py.generators.bus.isin(electricity_buses)) &
            (n_py.generators.bus.apply(lambda x: str(x)[:2] in eu27_countries))
        ]
        if not el_generators_eu27.empty:
            relevant_gen_cols = [col for col in el_generators_eu27.index if col in n_py.generators_t.p.columns]
            if relevant_gen_cols:
                # Sum positive generation (MWh) across time and generators
                pypsa_el_generation += (n_py.generators_t.p[relevant_gen_cols].sum(axis=1) * weightings).sum()
            else:
                print("Warning: No matching time-series data found for EU27 electricity generators.")

    # Storage Units discharging (contributing to supply)
    if hasattr(n_py, 'storage_units') and not n_py.storage_units.empty:
        # Filter storage units on electricity buses and in EU27 countries
        el_storage_eu27 = n_py.storage_units[
            (n_py.storage_units.bus.isin(electricity_buses)) &
            (n_py.storage_units.bus.apply(lambda x: str(x)[:2] in eu27_countries))
        ]
        if not el_storage_eu27.empty:
            relevant_storage_cols = [col for col in el_storage_eu27.index if col in n_py.storage_units_t.p_dispatch.columns]
            if relevant_storage_cols:
                # Sum positive discharge (MWh) across time and storage units
                pypsa_el_generation += (n_py.storage_units_t.p_dispatch[relevant_storage_cols].sum(axis=1) * weightings).sum()
            else:
                print("Warning: No matching time-series data found for EU27 electricity storage units.")

    pypsa_el_generation /= 1e6  # Convert MWh to TWh
else:
    print("Warning: No time-series generation data (generators_t.p or storage_units_t.p_dispatch) found in PyPSA network.")

print(f"PyPSA EU27 Annual Electricity Generation: {pypsa_el_generation:.2f} TWh")

# 3. Compare Electricity Generation
if validation_el_generation != 0:
    generation_delta = pypsa_el_generation - validation_el_generation
    generation_diff_percent = (generation_delta / validation_el_generation) * 100
    print(f"Delta Electricity Generation: {generation_delta:.2f} TWh ({generation_diff_percent:.2f} %)")
else:
    print("Cannot calculate delta or percentage for electricity generation as validation data is zero.")

=== EU27 Electricity Demand Comparison ===
Validation EU27 Electricity Demand: 0.00 TWh
PyPSA EU27 Electricity Demand: 3539.89 TWh
Cannot calculate delta or percentage for electricity demand as validation data is zero.

=== EU27 Annual Electricity Generation Comparison ===
Validation EU27 Annual Electricity Generation: 0.00 TWh
PyPSA EU27 Annual Electricity Generation: 4313.58 TWh
Cannot calculate delta or percentage for electricity generation as validation data is zero.
